# SS Model - Within 1 Day Diagnostic

This notebook trains simple tabular models for `SS_mg_l_clean` using Sentinel-2 features only.

Main condition:
- use `relaxed_v4` dataset
- keep only image-field matchups with `abs_date_diff_days <= 1`
- train years: 2562-2566
- test years: 2567-2568
- train on `log1p(SS)` and report metrics on both log scale and original mg/L scale


In [ ]:
# Imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge, LinearRegression, HuberRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Config
DATA_PATH = Path(r".\Data\tabular_data_14\train\v4\rswq_model_input_relaxed_s2_context_2562_2568.csv")
TARGET_COL = "SS_mg_l_clean"
TARGET_LOG_COL = "SS_log1p"
RANDOM_STATE = 19

TRAIN_YEAR_MIN = 2562
TRAIN_YEAR_MAX = 2566
TEST_YEAR_MIN = 2567
TEST_YEAR_MAX = 2568
MAX_MATCH_DAYS = 1

# Edit this list if you want to test another feature set.
# NIR uses Sentinel-2 B8_median here.
FEATURE_COLS = [
    "B1_median", "B2_median", "B3_median", "B4_median",
    "B5_median", "B6_median", "B7_median", "B8_median",
    "B8A_median", "B9_median", "B11_median", "B12_median",
    "NDTI_median", "NDWI_median",
    "NDWI1", "NDWI3",
    "B4_B3_ratio", "B8_B4_ratio",

]


In [ ]:
# Load data and create the within-1-day temporal split.
df = pd.read_csv(DATA_PATH, low_memory=False)
df[TARGET_LOG_COL] = np.log1p(pd.to_numeric(df[TARGET_COL], errors="coerce"))

# Custom Sentinel-2 water-index and ratio features.
# These are derived from existing band medians, so they are safe to recompute after loading the CSV.
# NIR uses Sentinel-2 B8_median. Therefore B8_B4_ratio and NIR_Red_ratio are intentionally equivalent.
CUSTOM_S2_INDEX_COLS = [
    "NDWI1", "NDWI2", "NDWI3",
    "B4_B3_ratio", "B8_B4_ratio",
    "NIR_Blue_ratio", "NIR_Green_ratio", "NIR_Red_ratio",
    "Red_NIR_ND",
]


def normalized_difference(numerator_a, numerator_b, eps=1e-12):
    a = pd.to_numeric(numerator_a, errors="coerce")
    b = pd.to_numeric(numerator_b, errors="coerce")
    denom = a + b
    out = np.where(np.abs(denom) > eps, (a - b) / denom, np.nan)
    return np.where(np.isfinite(out), out, np.nan)


def safe_ratio(numerator, denominator, eps=1e-12):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    out = np.where(np.abs(den) > eps, num / den, np.nan)
    return np.where(np.isfinite(out), out, np.nan)


required_custom_band_cols = ["B2_median", "B3_median", "B4_median", "B8_median", "B11_median", "B12_median"]
missing_custom_band_cols = [c for c in required_custom_band_cols if c not in df.columns]
if missing_custom_band_cols:
    raise KeyError(f"Missing bands needed for custom S2 features: {missing_custom_band_cols}")

blue = df["B2_median"]
green = df["B3_median"]
red = df["B4_median"]
nir = df["B8_median"]

# Existing custom NDWI variants.
df["NDWI1"] = normalized_difference(green, nir)
df["NDWI2"] = normalized_difference(nir, df["B11_median"])
df["NDWI3"] = normalized_difference(nir, df["B12_median"])

# Suspended-solid/turbidity-oriented band ratios requested for testing.
df["B4_B3_ratio"] = safe_ratio(red, green)
df["B8_B4_ratio"] = safe_ratio(nir, red)
df["NIR_Blue_ratio"] = safe_ratio(nir, blue)
df["NIR_Green_ratio"] = safe_ratio(nir, green)
df["NIR_Red_ratio"] = safe_ratio(nir, red)
df["Red_NIR_ND"] = normalized_difference(red, nir)

missing_features = [c for c in FEATURE_COLS if c not in df.columns]
if missing_features:
    raise KeyError(f"Missing feature columns: {missing_features}")

match_days = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
year = pd.to_numeric(df["year_be"], errors="coerce")
base_mask = df[TARGET_COL].notna() & match_days.le(MAX_MATCH_DAYS)

train_raw = df.loc[base_mask & year.between(TRAIN_YEAR_MIN, TRAIN_YEAR_MAX)].copy()
test_df = df.loc[base_mask & year.between(TEST_YEAR_MIN, TEST_YEAR_MAX)].copy()

split_summary = pd.DataFrame({
    "split": ["train_raw", "test"],
    "rows": [len(train_raw), len(test_df)],
    "stations": [train_raw["station_canonical"].nunique(), test_df["station_canonical"].nunique()],
    "target_median": [train_raw[TARGET_COL].median(), test_df[TARGET_COL].median()],
    "target_p95": [train_raw[TARGET_COL].quantile(0.95), test_df[TARGET_COL].quantile(0.95)],
    "target_max": [train_raw[TARGET_COL].max(), test_df[TARGET_COL].max()],
})

custom_index_summary = df[CUSTOM_S2_INDEX_COLS].describe().T[["count", "mean", "std", "min", "max"]]

display(split_summary.round(4))
print("Custom S2 indices/ratios added:", CUSTOM_S2_INDEX_COLS)
display(custom_index_summary.round(4))
print(f"Dataset: {DATA_PATH}")
print(f"Feature count: {len(FEATURE_COLS)}")
print(f"Match condition: abs_date_diff_days <= {MAX_MATCH_DAYS}")


In [ ]:
# Models and metrics.
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

try:
    from cubist import Cubist
    HAS_CUBIST = True
except Exception as exc:
    HAS_CUBIST = False
    print("Cubist not available:", exc)


def make_models(random_state=42):
    models = {
        "DummyMedian": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor(strategy="median")),
        ]),
        "LinearRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]),
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "ElasticNet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.01, l1_ratio=0.15, max_iter=20000, random_state=random_state)),
        ]),
        "BayesianRidge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", BayesianRidge()),
        ]),
        "HuberRegressor": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000)),
        ]),
    }

    # Linear support-vector models are useful when samples are few and spectral features are correlated.
    for c in [0.3, 0.5, 1.0, 2.0]:
        for epsilon in [0.0, 0.05, 0.10]:
            models[f"LinearSVR_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LinearSVR(C=c, epsilon=epsilon, random_state=random_state, max_iter=30000)),
            ])

    # Nonlinear SVR-RBF is a common water-quality baseline for small remote-sensing matchup tables.
    for c in [0.3, 1.0, 3.0, 10.0]:
        for epsilon in [0.05, 0.10, 0.20]:
            models[f"SVR_RBF_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", SVR(kernel="rbf", C=c, epsilon=epsilon, gamma="scale")),
            ])

    for n_components in [2, 3, 5, 8]:
        models[f"PLS_{n_components}"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=n_components)),
        ])

    models.update({
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        ]),
        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, l2_regularization=0.1, random_state=random_state)),
        ]),
    })

    if HAS_XGBOOST:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=500, max_depth=2, learning_rate=0.03,
                subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
                objective="reg:squarederror", random_state=random_state, n_jobs=-1,
            )),
        ])

    if HAS_CATBOOST:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=700, depth=3, learning_rate=0.03,
                loss_function="RMSE", random_seed=random_state,
                verbose=False, allow_writing_files=False,
            )),
        ])

    if HAS_CUBIST:
        models["Cubist_c1"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=1, random_state=random_state)),
        ])
        models["Cubist_c5"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=5, random_state=random_state)),
        ])
        models["Cubist_c5_n3"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=5, neighbors=3, random_state=random_state)),
        ])

    return models

def safe_corr(corr_func, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    if ok.sum() < 3 or np.std(y_true[ok]) == 0 or np.std(y_pred[ok]) == 0:
        return np.nan
    return float(corr_func(y_true[ok], y_pred[ok])[0])


def regression_metrics(y_log, pred_log, y_original):
    pred_log = np.asarray(pred_log, dtype=float).ravel()
    pred_original = np.expm1(np.clip(pred_log, 0, None))
    y_original = np.asarray(y_original, dtype=float)
    return {
        "log_r2": r2_score(y_log, pred_log),
        "log_rmse": np.sqrt(mean_squared_error(y_log, pred_log)),
        "original_r2": r2_score(y_original, pred_original),
        "original_rmse": np.sqrt(mean_squared_error(y_original, pred_original)),
        "original_mae": mean_absolute_error(y_original, pred_original),
        "spearman_r": safe_corr(spearmanr, y_original, pred_original),
        "pearson_r": safe_corr(pearsonr, y_original, pred_original),
        "bias": float(np.mean(pred_original - y_original)),
    }


def log_iqr_train_filter(train_data):
    y = train_data[TARGET_LOG_COL]
    q1 = y.quantile(0.25)
    q3 = y.quantile(0.75)
    iqr = q3 - q1
    keep = y.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return train_data.loc[keep].copy(), train_data.loc[~keep].copy()


In [ ]:
# Train and evaluate.
train_logiqr, removed_logiqr = log_iqr_train_filter(train_raw)
train_configs = {
    "within1_no_cut": train_raw,
    "within1_train_log_iqr": train_logiqr,
}

rows = []
prediction_tables = {}

for config_name, train_df in train_configs.items():
    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_LOG_COL]
    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_LOG_COL]

    removed_train = len(train_raw) - len(train_df)

    for model_name, model in make_models(RANDOM_STATE).items():
        model.fit(X_train, y_train)
        pred_test_log = np.asarray(model.predict(X_test)).ravel()
        metrics = regression_metrics(y_test, pred_test_log, test_df[TARGET_COL])

        rows.append({
            "config": config_name,
            "model": model_name,
            "n_train": len(train_df),
            "n_test": len(test_df),
            "removed_train": removed_train,
            **metrics,
        })

        prediction_tables[(config_name, model_name)] = pd.DataFrame({
            "sample_id": test_df["sample_id"].values,
            "station_canonical": test_df["station_canonical"].values,
            "year_be": test_df["year_be"].values,
            "observed": test_df[TARGET_COL].values,
            "predicted": np.expm1(np.clip(pred_test_log, 0, None)),
        })

results = pd.DataFrame(rows).sort_values(["original_r2", "log_r2"], ascending=False).reset_index(drop=True)

show_cols = [
    "config", "model", "n_train", "n_test", "removed_train",
    "original_r2", "original_rmse", "original_mae",
    "log_r2", "log_rmse", "spearman_r", "pearson_r", "bias",
]

display(results[show_cols].round(4))


In [ ]:
# Plot best model on original scale.
best = results.iloc[0]
best_key = (best["config"], best["model"])
best_pred = prediction_tables[best_key].copy()
best_pred["residual"] = best_pred["predicted"] - best_pred["observed"]

print("Best config/model:", best_key)
display(best_pred.sort_values("observed", ascending=False).head(12).round(4))

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(best_pred["observed"], best_pred["predicted"], alpha=0.8)
limit = max(best_pred["observed"].max(), best_pred["predicted"].max()) * 1.05
ax.plot([0, limit], [0, limit], linestyle="--", color="gray", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("Observed SS (mg/L)")
ax.set_ylabel("Predicted SS (mg/L)")
ax.set_title(f"{best['config']} | {best['model']} | R2={best['original_r2']:.3f}")
ax.grid(True, alpha=0.3)
plt.show()


## Two-Stage Low/Censored SS Experiment

This experiment treats `SS <= 10 mg/L` as a low/censored group.

Workflow:
1. classify each sample as low/censored (`SS <= 10`) or high (`SS > 10`)
2. train regression only on high samples
3. combine predictions: low samples get the train low median, high samples get the high-regression prediction

`ORACLE_true_low_high` is included only as a diagnostic upper bound. It uses the true test low/high label and should not be reported as a deployable model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

LOW_THRESHOLD = 10.0

train_stage = train_raw.copy()
test_stage = test_df.copy()
train_stage["is_high"] = (train_stage[TARGET_COL] > LOW_THRESHOLD).astype(int)
test_stage["is_high"] = (test_stage[TARGET_COL] > LOW_THRESHOLD).astype(int)

low_value = train_stage.loc[train_stage["is_high"].eq(0), TARGET_COL].median()
high_train = train_stage.loc[train_stage["is_high"].eq(1)].copy()

print("Low/censored threshold:", LOW_THRESHOLD)
print("Low prediction value from train median:", round(low_value, 4))
print("Train low/high:", train_stage["is_high"].value_counts().sort_index().to_dict())
print("Test low/high :", test_stage["is_high"].value_counts().sort_index().to_dict())


def make_classifiers(random_state=42):
    classifiers = {
        "LogisticBalanced": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=5000, random_state=random_state)),
        ]),
        "SVC_RBF_balanced": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=1.0, gamma="scale", class_weight="balanced", random_state=random_state)),
        ]),
        "RandomForestCls": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced", random_state=random_state, n_jobs=-1)),
        ]),
        "ExtraTreesCls": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced", random_state=random_state, n_jobs=-1)),
        ]),
        "HistGBCls": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.04, l2_regularization=0.1, random_state=random_state)),
        ]),
    }
    if HAS_XGBOOST and XGBClassifier is not None:
        classifiers["XGBCls"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=300, max_depth=2, learning_rate=0.03,
                subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
                eval_metric="logloss", random_state=random_state, n_jobs=-1,
            )),
        ])
    if HAS_CATBOOST:
        classifiers["CatBoostCls"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostClassifier(
                iterations=500, depth=3, learning_rate=0.03,
                loss_function="Logloss", auto_class_weights="Balanced",
                random_seed=random_state, verbose=False, allow_writing_files=False,
            )),
        ])
    return classifiers


def make_high_regressors(random_state=42):
    regressors = {
        "LinearRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]),
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "ElasticNet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.01, l1_ratio=0.15, max_iter=20000, random_state=random_state)),
        ]),
        "BayesianRidge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", BayesianRidge()),
        ]),
        "HuberRegressor": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000)),
        ]),
    }

    for c in [0.3, 0.5, 1.0, 2.0]:
        for epsilon in [0.0, 0.05, 0.10]:
            regressors[f"LinearSVR_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LinearSVR(C=c, epsilon=epsilon, random_state=random_state, max_iter=30000)),
            ])

    for c in [0.3, 1.0, 3.0, 10.0]:
        for epsilon in [0.05, 0.10, 0.20]:
            regressors[f"SVR_RBF_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", SVR(kernel="rbf", C=c, epsilon=epsilon, gamma="scale")),
            ])

    for n_components in [2, 3, 5, 8]:
        regressors[f"PLS_{n_components}"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=n_components)),
        ])

    regressors.update({
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        ]),
    })

    if HAS_XGBOOST:
        regressors["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=500, max_depth=2, learning_rate=0.03,
                subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
                objective="reg:squarederror", random_state=random_state, n_jobs=-1,
            )),
        ])
    if HAS_CATBOOST:
        regressors["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=700, depth=3, learning_rate=0.03,
                loss_function="RMSE", random_seed=random_state,
                verbose=False, allow_writing_files=False,
            )),
        ])
    if HAS_CUBIST:
        regressors["Cubist_c1"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=1, random_state=random_state)),
        ])
        regressors["Cubist_c5"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=5, random_state=random_state)),
        ])
        regressors["Cubist_c5_n3"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=5, neighbors=3, random_state=random_state)),
        ])
    return regressors

def metrics_from_original(y_true, y_pred):
    return {
        "original_r2": r2_score(y_true, y_pred),
        "original_rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "original_mae": mean_absolute_error(y_true, y_pred),
        "spearman_r": safe_corr(spearmanr, y_true, y_pred),
        "pearson_r": safe_corr(pearsonr, y_true, y_pred),
        "bias": float(np.mean(y_pred - y_true)),
    }

X_train_stage = train_stage[FEATURE_COLS]
y_train_class = train_stage["is_high"]
X_test_stage = test_stage[FEATURE_COLS]
y_test_class = test_stage["is_high"]
y_test_original = test_stage[TARGET_COL].values

classifier_predictions = {}
classifier_rows = []
for classifier_name, classifier in make_classifiers(RANDOM_STATE).items():
    classifier.fit(X_train_stage, y_train_class)
    pred = np.asarray(classifier.predict(X_test_stage)).ravel()
    if classifier_name == "XGBCls":
        pred = (pred >= 0.5).astype(int)
    else:
        pred = pred.astype(int)
    classifier_predictions[classifier_name] = pred
    classifier_rows.append({
        "classifier": classifier_name,
        "accuracy": accuracy_score(y_test_class, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test_class, pred),
        "f1_high": f1_score(y_test_class, pred, pos_label=1),
        "pred_high": int(pred.sum()),
        "true_high": int(y_test_class.sum()),
    })

classifier_results = pd.DataFrame(classifier_rows).sort_values(["balanced_accuracy", "accuracy"], ascending=False)
display(classifier_results.round(4))

regressor_predictions = {}
for regressor_name, regressor in make_high_regressors(RANDOM_STATE).items():
    regressor.fit(high_train[FEATURE_COLS], high_train[TARGET_LOG_COL])
    pred_log = np.asarray(regressor.predict(X_test_stage)).ravel()
    regressor_predictions[regressor_name] = np.expm1(np.clip(pred_log, 0, None))

rows = []
for classifier_name, high_pred in classifier_predictions.items():
    for regressor_name, reg_pred in regressor_predictions.items():
        combined_pred = np.where(high_pred == 1, reg_pred, low_value)
        row = {
            "classifier": classifier_name,
            "regressor": regressor_name,
            "n_train_classifier": len(train_stage),
            "n_train_regressor_high_only": len(high_train),
            "n_test": len(test_stage),
            "pred_high": int(high_pred.sum()),
        }
        row.update(metrics_from_original(y_test_original, combined_pred))
        rows.append(row)

# Diagnostic upper bound: assumes the low/high classifier is perfect on test.
true_high = y_test_class.values
for regressor_name, reg_pred in regressor_predictions.items():
    combined_pred = np.where(true_high == 1, reg_pred, low_value)
    row = {
        "classifier": "ORACLE_true_low_high",
        "regressor": regressor_name,
        "n_train_classifier": len(train_stage),
        "n_train_regressor_high_only": len(high_train),
        "n_test": len(test_stage),
        "pred_high": int(true_high.sum()),
    }
    row.update(metrics_from_original(y_test_original, combined_pred))
    rows.append(row)

two_stage_results = pd.DataFrame(rows).sort_values(["original_r2", "spearman_r"], ascending=False).reset_index(drop=True)

display(two_stage_results[[
    "classifier", "regressor", "n_train_classifier", "n_train_regressor_high_only", "n_test", "pred_high",
    "original_r2", "original_rmse", "original_mae", "spearman_r", "pearson_r", "bias",
]].round(4).head(30))


## Sensitivity: Direct SS Threshold Outlier Cut

This sensitivity experiment removes SS outliers using an original-scale threshold in mg/L, not a log-IQR rule.

Edit the config cell below to choose:
- `DIRECT_SS_MAX_MATCH_DAYS`: Sentinel-2 match window in days, for example 0, 1, 3, or 7
- `DIRECT_SS_MAX_MG_L`: maximum SS kept, for example 50, 75, 100, 150, 200, 300, 500, or `None`
- `DIRECT_SS_CUT_SCOPE`: `train_test` for a cleaned-range diagnostic, or `train_only` to keep the full test range

The section reports both the normal one-stage log-regression models and the two-stage low/censored SS setup.


In [ ]:
# Direct SS threshold experiment config.
# Change these values and rerun this section only.

DIRECT_SS_MAX_MATCH_DAYS = 1
DIRECT_SS_MAX_MG_L = 130.0      # Set to None to disable direct threshold filtering.
DIRECT_SS_CUT_SCOPE = "train_test"  # options: "train_test" or "train_only"
DIRECT_SS_LOW_THRESHOLD = LOW_THRESHOLD if "LOW_THRESHOLD" in globals() else 10.0
DIRECT_SS_RANDOM_STATE = RANDOM_STATE
DIRECT_SS_FEATURE_COLS = list(FEATURE_COLS)

if DIRECT_SS_CUT_SCOPE not in {"train_test", "train_only"}:
    raise ValueError("DIRECT_SS_CUT_SCOPE must be 'train_test' or 'train_only'.")
if not DIRECT_SS_FEATURE_COLS:
    raise ValueError("DIRECT_SS_FEATURE_COLS is empty. Fill FEATURE_COLS first, then rerun this section.")

print("Direct SS threshold experiment config")
print(f"Match window: abs_date_diff_days <= {DIRECT_SS_MAX_MATCH_DAYS}")
print(f"SS threshold: {DIRECT_SS_MAX_MG_L} mg/L")
print(f"Cut scope: {DIRECT_SS_CUT_SCOPE}")
print(f"Low/censored threshold for two-stage model: SS <= {DIRECT_SS_LOW_THRESHOLD} mg/L")
print(f"Feature count: {len(DIRECT_SS_FEATURE_COLS)}")


In [ ]:
# Count how many SS rows remain under common direct thresholds and match windows.

ss_threshold_grid = [50, 75, 100, 150, 200, 300, 500, None]
window_grid = [0, 1, 3, 7]

year_num = pd.to_numeric(df["year_be"], errors="coerce")
diff_num = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
target_num = pd.to_numeric(df[TARGET_COL], errors="coerce")

count_rows = []
for max_days in window_grid:
    base = target_num.notna() & diff_num.le(max_days)
    train_raw_mask = base & year_num.between(TRAIN_YEAR_MIN, TRAIN_YEAR_MAX)
    test_raw_mask = base & year_num.between(TEST_YEAR_MIN, TEST_YEAR_MAX)

    for threshold in ss_threshold_grid:
        if threshold is None:
            keep = pd.Series(True, index=df.index)
            threshold_label = "none"
        else:
            keep = target_num.le(float(threshold))
            threshold_label = str(threshold)

        train_keep = train_raw_mask & keep
        test_keep = test_raw_mask & keep
        count_rows.append({
            "max_match_days": max_days,
            "threshold_mg_l": threshold_label,
            "train_before": int(train_raw_mask.sum()),
            "train_after": int(train_keep.sum()),
            "train_removed": int((train_raw_mask & ~keep).sum()),
            "test_before": int(test_raw_mask.sum()),
            "test_after": int(test_keep.sum()),
            "test_removed": int((test_raw_mask & ~keep).sum()),
            "train_max_after": target_num.loc[train_keep].max(),
            "test_max_after": target_num.loc[test_keep].max(),
        })

direct_ss_threshold_count_table = pd.DataFrame(count_rows)
display(direct_ss_threshold_count_table)


In [ ]:
# Build train/test data for the selected direct SS threshold experiment.

required_direct_cols = ["year_be", "abs_date_diff_days", TARGET_COL, TARGET_LOG_COL] + DIRECT_SS_FEATURE_COLS
missing_direct_cols = [c for c in required_direct_cols if c not in df.columns]
if missing_direct_cols:
    raise KeyError(f"Missing columns for direct SS threshold experiment: {missing_direct_cols}")

if TARGET_LOG_COL not in df.columns:
    df[TARGET_LOG_COL] = np.log1p(pd.to_numeric(df[TARGET_COL], errors="coerce"))

year_num = pd.to_numeric(df["year_be"], errors="coerce")
diff_num = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
target_num = pd.to_numeric(df[TARGET_COL], errors="coerce")

match_mask = target_num.notna() & diff_num.le(DIRECT_SS_MAX_MATCH_DAYS)
train_raw_mask = match_mask & year_num.between(TRAIN_YEAR_MIN, TRAIN_YEAR_MAX)
test_raw_mask = match_mask & year_num.between(TEST_YEAR_MIN, TEST_YEAR_MAX)

if DIRECT_SS_MAX_MG_L is None:
    threshold_keep = pd.Series(True, index=df.index)
else:
    threshold_keep = target_num.le(float(DIRECT_SS_MAX_MG_L))

direct_ss_train_raw_df = df.loc[train_raw_mask].copy()
direct_ss_test_raw_df = df.loc[test_raw_mask].copy()
direct_ss_train_model_df = df.loc[train_raw_mask & threshold_keep].copy()

if DIRECT_SS_CUT_SCOPE == "train_test":
    direct_ss_test_eval_df = df.loc[test_raw_mask & threshold_keep].copy()
else:
    direct_ss_test_eval_df = direct_ss_test_raw_df.copy()

direct_ss_removed_train_df = df.loc[train_raw_mask & ~threshold_keep].copy()
direct_ss_removed_test_df = df.loc[test_raw_mask & ~threshold_keep].copy()

def direct_ss_split_stats(name, data):
    return {
        "condition": name,
        "rows": len(data),
        "stations": data["station_canonical"].nunique() if "station_canonical" in data.columns else np.nan,
        "target_median": data[TARGET_COL].median(),
        "target_p95": data[TARGET_COL].quantile(0.95),
        "target_max": data[TARGET_COL].max(),
        "low_or_censored_count": int(data[TARGET_COL].le(DIRECT_SS_LOW_THRESHOLD).sum()) if len(data) else 0,
        "high_count": int(data[TARGET_COL].gt(DIRECT_SS_LOW_THRESHOLD).sum()) if len(data) else 0,
    }

direct_ss_threshold_summary = pd.DataFrame([
    direct_ss_split_stats("train_before_threshold", direct_ss_train_raw_df),
    direct_ss_split_stats("train_after_threshold", direct_ss_train_model_df),
    direct_ss_split_stats("test_before_threshold", direct_ss_test_raw_df),
    direct_ss_split_stats("test_eval_after_threshold" if DIRECT_SS_CUT_SCOPE == "train_test" else "test_eval_unchanged", direct_ss_test_eval_df),
])

print("Selected direct SS threshold experiment")
print(f"Train years: {TRAIN_YEAR_MIN}-{TRAIN_YEAR_MAX}")
print(f"Test years : {TEST_YEAR_MIN}-{TEST_YEAR_MAX}")
print(f"Match window: <= {DIRECT_SS_MAX_MATCH_DAYS} days")
print(f"Threshold: {DIRECT_SS_MAX_MG_L} mg/L")
print(f"Train rows removed: {len(direct_ss_removed_train_df)}")
print(f"Test rows removed : {len(direct_ss_removed_test_df) if DIRECT_SS_CUT_SCOPE == 'train_test' else 0}")
display(direct_ss_threshold_summary.round(4))

review_cols = [c for c in [
    "sample_id", "station_canonical", "year_be", "field_date", "sat_date",
    TARGET_COL, "turbidity_NTU_clean", "match_window", "abs_date_diff_days",
    "valid_water_pixel_count", "water_pixel_fraction", "water_pixel_pct",
    "local_cloud_fraction", "local_cloud_pct",
    "rain_gsmap_prior_24h_mm", "rain_gsmap_prior_72h_mm", "rain_gsmap_prior_168h_mm",
    "era5h_runoff_prior_24h_mm", "era5h_runoff_prior_72h_mm", "era5h_runoff_prior_168h_mm",
] if c in df.columns]

if len(direct_ss_removed_train_df):
    print("Train rows removed by direct SS threshold")
    display(direct_ss_removed_train_df[review_cols].sort_values(TARGET_COL, ascending=False).round(4))
else:
    print("No train rows removed by the selected direct SS threshold.")

if DIRECT_SS_CUT_SCOPE == "train_test" and len(direct_ss_removed_test_df):
    print("Test rows removed by direct SS threshold")
    display(direct_ss_removed_test_df[review_cols].sort_values(TARGET_COL, ascending=False).round(4))
elif DIRECT_SS_CUT_SCOPE == "train_test":
    print("No test rows removed by the selected direct SS threshold.")
else:
    print("Test set is unchanged because DIRECT_SS_CUT_SCOPE = 'train_only'.")


In [ ]:
# One-stage log-regression models on the selected direct-threshold SS dataset.

if len(direct_ss_train_model_df) < 10:
    raise ValueError("Too few train rows after direct SS threshold filtering.")
if len(direct_ss_test_eval_df) < 5:
    raise ValueError("Too few test rows after direct SS threshold filtering.")

X_direct_ss_train = direct_ss_train_model_df[DIRECT_SS_FEATURE_COLS].copy()
X_direct_ss_test = direct_ss_test_eval_df[DIRECT_SS_FEATURE_COLS].copy()
y_direct_ss_train_log = direct_ss_train_model_df[TARGET_LOG_COL].copy()
y_direct_ss_test_log = direct_ss_test_eval_df[TARGET_LOG_COL].copy()
y_direct_ss_train_original = direct_ss_train_model_df[TARGET_COL].copy()
y_direct_ss_test_original = direct_ss_test_eval_df[TARGET_COL].copy()

direct_ss_regression_rows = []
direct_ss_regression_predictions = {}

for model_name, model in make_models(DIRECT_SS_RANDOM_STATE).items():
    model.fit(X_direct_ss_train, y_direct_ss_train_log)
    pred_train_log = np.asarray(model.predict(X_direct_ss_train)).ravel()
    pred_test_log = np.asarray(model.predict(X_direct_ss_test)).ravel()

    train_metrics = regression_metrics(y_direct_ss_train_log, pred_train_log, y_direct_ss_train_original)
    test_metrics = regression_metrics(y_direct_ss_test_log, pred_test_log, y_direct_ss_test_original)

    row = {
        "approach": "one_stage_log_regression",
        "model": model_name,
        "n_features": len(DIRECT_SS_FEATURE_COLS),
        "n_train": len(X_direct_ss_train),
        "n_test": len(X_direct_ss_test),
        "max_match_days": DIRECT_SS_MAX_MATCH_DAYS,
        "threshold_mg_l": DIRECT_SS_MAX_MG_L if DIRECT_SS_MAX_MG_L is not None else "none",
        "cut_scope": DIRECT_SS_CUT_SCOPE,
        "train_removed_by_threshold": len(direct_ss_removed_train_df),
        "test_removed_by_threshold": len(direct_ss_removed_test_df) if DIRECT_SS_CUT_SCOPE == "train_test" else 0,
    }
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"test_{k}": v for k, v in test_metrics.items()})
    direct_ss_regression_rows.append(row)

    pred_original = np.expm1(np.clip(pred_test_log, 0, None))
    direct_ss_regression_predictions[model_name] = pd.DataFrame({
        "sample_id": direct_ss_test_eval_df["sample_id"].values if "sample_id" in direct_ss_test_eval_df.columns else np.arange(len(direct_ss_test_eval_df)),
        "station_canonical": direct_ss_test_eval_df["station_canonical"].values if "station_canonical" in direct_ss_test_eval_df.columns else np.nan,
        "year_be": direct_ss_test_eval_df["year_be"].values,
        "observed_original": y_direct_ss_test_original.values,
        "predicted_original": pred_original,
        "observed_log": y_direct_ss_test_log.values,
        "predicted_log": np.clip(pred_test_log, 0, None),
        "residual_original": pred_original - y_direct_ss_test_original.values,
    })

direct_ss_regression_results = pd.DataFrame(direct_ss_regression_rows).sort_values(
    ["test_original_r2", "test_log_r2"], ascending=False
).reset_index(drop=True)

direct_ss_show_cols = [
    "approach", "model", "n_features", "n_train", "n_test", "max_match_days", "threshold_mg_l", "cut_scope",
    "train_removed_by_threshold", "test_removed_by_threshold",
    "test_original_r2", "test_original_rmse", "test_original_mae",
    "test_log_r2", "test_log_rmse", "test_spearman_r", "test_pearson_r", "test_bias",
    "train_original_r2", "train_log_r2",
]

print("Direct SS threshold: one-stage regression results")
display(direct_ss_regression_results[direct_ss_show_cols].round(4))


In [ ]:
# Two-stage low/censored SS models on the selected direct-threshold dataset.

def direct_ss_metrics_from_original_with_log(y_true_original, y_pred_original):
    y_true_original = np.asarray(y_true_original, dtype=float)
    y_pred_original = np.asarray(y_pred_original, dtype=float)
    y_pred_original = np.clip(y_pred_original, 0, None)
    out = metrics_from_original(y_true_original, y_pred_original)
    y_true_log = np.log1p(y_true_original)
    y_pred_log = np.log1p(y_pred_original)
    out["log_r2"] = r2_score(y_true_log, y_pred_log)
    out["log_rmse"] = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    return out

# Rebuild low/high labels after the direct threshold split.
direct_ss_train_stage = direct_ss_train_model_df.copy()
direct_ss_test_stage = direct_ss_test_eval_df.copy()
direct_ss_train_stage["is_high"] = direct_ss_train_stage[TARGET_COL].gt(DIRECT_SS_LOW_THRESHOLD).astype(int)
direct_ss_test_stage["is_high"] = direct_ss_test_stage[TARGET_COL].gt(DIRECT_SS_LOW_THRESHOLD).astype(int)

direct_ss_low_value = direct_ss_train_stage.loc[direct_ss_train_stage["is_high"].eq(0), TARGET_COL].median()
direct_ss_high_train = direct_ss_train_stage.loc[direct_ss_train_stage["is_high"].eq(1)].copy()

if direct_ss_train_stage["is_high"].nunique() < 2:
    raise ValueError("Two-stage classifier needs both low/censored and high SS classes in train.")
if len(direct_ss_high_train) < 10:
    raise ValueError("Too few high-SS train rows for the stage-2 regressor.")

X_direct_stage_train = direct_ss_train_stage[DIRECT_SS_FEATURE_COLS]
y_direct_stage_class = direct_ss_train_stage["is_high"]
X_direct_stage_test = direct_ss_test_stage[DIRECT_SS_FEATURE_COLS]
y_direct_stage_test_class = direct_ss_test_stage["is_high"]
y_direct_stage_test_original = direct_ss_test_stage[TARGET_COL].values

print("Direct SS two-stage class balance")
print("Train low/high:", direct_ss_train_stage["is_high"].value_counts().sort_index().to_dict())
print("Test low/high :", direct_ss_test_stage["is_high"].value_counts().sort_index().to_dict())
print("Low prediction value from train median:", round(float(direct_ss_low_value), 4))

classifier_predictions = {}
classifier_rows = []
for classifier_name, classifier in make_classifiers(DIRECT_SS_RANDOM_STATE).items():
    classifier.fit(X_direct_stage_train, y_direct_stage_class)
    pred_class = np.asarray(classifier.predict(X_direct_stage_test)).ravel().astype(int)
    classifier_predictions[classifier_name] = pred_class
    classifier_rows.append({
        "classifier": classifier_name,
        "accuracy": accuracy_score(y_direct_stage_test_class, pred_class),
        "balanced_accuracy": balanced_accuracy_score(y_direct_stage_test_class, pred_class),
        "f1_high": f1_score(y_direct_stage_test_class, pred_class, pos_label=1),
        "pred_high": int(pred_class.sum()),
        "true_high": int(y_direct_stage_test_class.sum()),
    })

direct_ss_classifier_results = pd.DataFrame(classifier_rows).sort_values(
    ["balanced_accuracy", "accuracy"], ascending=False
).reset_index(drop=True)
print("Direct SS two-stage classifier results")
display(direct_ss_classifier_results.round(4))

regressor_predictions = {}
for regressor_name, regressor in make_high_regressors(DIRECT_SS_RANDOM_STATE).items():
    regressor.fit(direct_ss_high_train[DIRECT_SS_FEATURE_COLS], direct_ss_high_train[TARGET_LOG_COL])
    pred_log = np.asarray(regressor.predict(X_direct_stage_test)).ravel()
    regressor_predictions[regressor_name] = np.expm1(np.clip(pred_log, 0, None))

rows = []
direct_ss_two_stage_predictions = {}
for classifier_name, pred_high in classifier_predictions.items():
    for regressor_name, pred_reg in regressor_predictions.items():
        combined_pred = np.where(pred_high == 1, pred_reg, direct_ss_low_value)
        metrics = direct_ss_metrics_from_original_with_log(y_direct_stage_test_original, combined_pred)
        row = {
            "approach": "two_stage_low_high",
            "classifier": classifier_name,
            "regressor": regressor_name,
            "model": f"{classifier_name} + {regressor_name}",
            "n_features": len(DIRECT_SS_FEATURE_COLS),
            "n_train": len(direct_ss_train_stage),
            "n_train_high_only": len(direct_ss_high_train),
            "n_test": len(direct_ss_test_stage),
            "max_match_days": DIRECT_SS_MAX_MATCH_DAYS,
            "threshold_mg_l": DIRECT_SS_MAX_MG_L if DIRECT_SS_MAX_MG_L is not None else "none",
            "cut_scope": DIRECT_SS_CUT_SCOPE,
            "train_removed_by_threshold": len(direct_ss_removed_train_df),
            "test_removed_by_threshold": len(direct_ss_removed_test_df) if DIRECT_SS_CUT_SCOPE == "train_test" else 0,
            "pred_high": int(pred_high.sum()),
        }
        row.update(metrics)
        rows.append(row)
        direct_ss_two_stage_predictions[(classifier_name, regressor_name)] = pd.DataFrame({
            "sample_id": direct_ss_test_stage["sample_id"].values if "sample_id" in direct_ss_test_stage.columns else np.arange(len(direct_ss_test_stage)),
            "station_canonical": direct_ss_test_stage["station_canonical"].values if "station_canonical" in direct_ss_test_stage.columns else np.nan,
            "year_be": direct_ss_test_stage["year_be"].values,
            "observed_original": y_direct_stage_test_original,
            "predicted_original": combined_pred,
            "observed_log": np.log1p(y_direct_stage_test_original),
            "predicted_log": np.log1p(np.clip(combined_pred, 0, None)),
            "residual_original": combined_pred - y_direct_stage_test_original,
            "predicted_high": pred_high,
            "true_high": y_direct_stage_test_class.values,
        })

# ORACLE is diagnostic only; it uses true test low/high labels.
true_high = y_direct_stage_test_class.values
for regressor_name, pred_reg in regressor_predictions.items():
    combined_pred = np.where(true_high == 1, pred_reg, direct_ss_low_value)
    metrics = direct_ss_metrics_from_original_with_log(y_direct_stage_test_original, combined_pred)
    row = {
        "approach": "oracle_two_stage_diagnostic_only",
        "classifier": "ORACLE_true_low_high",
        "regressor": regressor_name,
        "model": f"ORACLE_true_low_high + {regressor_name}",
        "n_features": len(DIRECT_SS_FEATURE_COLS),
        "n_train": len(direct_ss_train_stage),
        "n_train_high_only": len(direct_ss_high_train),
        "n_test": len(direct_ss_test_stage),
        "max_match_days": DIRECT_SS_MAX_MATCH_DAYS,
        "threshold_mg_l": DIRECT_SS_MAX_MG_L if DIRECT_SS_MAX_MG_L is not None else "none",
        "cut_scope": DIRECT_SS_CUT_SCOPE,
        "train_removed_by_threshold": len(direct_ss_removed_train_df),
        "test_removed_by_threshold": len(direct_ss_removed_test_df) if DIRECT_SS_CUT_SCOPE == "train_test" else 0,
        "pred_high": int(true_high.sum()),
    }
    row.update(metrics)
    rows.append(row)

direct_ss_two_stage_results = pd.DataFrame(rows).sort_values(
    ["original_r2", "log_r2"], ascending=False
).reset_index(drop=True)

two_stage_show_cols = [
    "approach", "model", "n_features", "n_train", "n_train_high_only", "n_test",
    "max_match_days", "threshold_mg_l", "cut_scope", "pred_high",
    "original_r2", "original_rmse", "original_mae", "log_r2", "log_rmse",
    "spearman_r", "pearson_r", "bias",
]

print("Direct SS threshold: two-stage results")
display(direct_ss_two_stage_results[two_stage_show_cols].round(4).head(40))


In [ ]:
# Compare the best actual direct-threshold one-stage and two-stage SS models.

best_one_stage = direct_ss_regression_results.iloc[0].copy()
actual_two_stage = direct_ss_two_stage_results.loc[
    direct_ss_two_stage_results["approach"].eq("two_stage_low_high")
].copy()
best_two_stage = actual_two_stage.iloc[0].copy() if not actual_two_stage.empty else None

summary_rows = [{
    "approach": "one_stage_log_regression",
    "model": best_one_stage["model"],
    "R2": best_one_stage["test_original_r2"],
    "RMSE": best_one_stage["test_original_rmse"],
    "log_R2": best_one_stage["test_log_r2"],
    "log_RMSE": best_one_stage["test_log_rmse"],
    "spearman_r": best_one_stage["test_spearman_r"],
    "test_bias": best_one_stage["test_bias"],
    "n_train": best_one_stage["n_train"],
    "n_test": best_one_stage["n_test"],
}]

if best_two_stage is not None:
    summary_rows.append({
        "approach": "two_stage_low_high",
        "model": best_two_stage["model"],
        "R2": best_two_stage["original_r2"],
        "RMSE": best_two_stage["original_rmse"],
        "log_R2": best_two_stage["log_r2"],
        "log_RMSE": best_two_stage["log_rmse"],
        "spearman_r": best_two_stage["spearman_r"],
        "test_bias": best_two_stage["bias"],
        "n_train": best_two_stage["n_train"],
        "n_test": best_two_stage["n_test"],
    })

direct_ss_best_summary = pd.DataFrame(summary_rows).sort_values("R2", ascending=False).reset_index(drop=True)
print("Best actual models from the direct SS threshold sensitivity experiment")
display(direct_ss_best_summary.round(4))

best_actual = direct_ss_best_summary.iloc[0]
if best_actual["approach"] == "one_stage_log_regression":
    plot_pred = direct_ss_regression_predictions[best_actual["model"]].copy()
else:
    classifier_name, regressor_name = best_actual["model"].split(" + ", 1)
    plot_pred = direct_ss_two_stage_predictions[(classifier_name, regressor_name)].copy()

print("Best actual direct-threshold model:", best_actual["approach"], "|", best_actual["model"])
display(plot_pred.sort_values("observed_original", ascending=False).head(15).round(4))

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(plot_pred["observed_original"], plot_pred["predicted_original"], alpha=0.8)
limit = max(plot_pred["observed_original"].max(), plot_pred["predicted_original"].max()) * 1.05
ax.plot([0, limit], [0, limit], linestyle="--", color="gray", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("Observed SS (mg/L)")
ax.set_ylabel("Predicted SS (mg/L)")
ax.set_title(f"Direct threshold SS | {best_actual['model']} | R2={best_actual['R2']:.3f}")
ax.grid(True, alpha=0.3)
plt.show()

print("Largest absolute residuals")
plot_pred["abs_residual_original"] = plot_pred["residual_original"].abs()
display(plot_pred.sort_values("abs_residual_original", ascending=False).head(20).round(4))


## Locked Baseline Context Feature Experiment

This section keeps the model setup fixed and changes only the feature set.

Locked setup:
- Match window: `abs_date_diff_days <= 1`
- SS range: `SS <= 130 mg/L`
- Stage 1 classifier: `XGBClassifier`
- Stage 2 regressor: `Ridge` on `log1p(SS)` using only high-SS training samples
- Low/high threshold: `SS > 10 mg/L`
- Temporal split: train `2562-2566`, test `2567-2568`

Purpose: test whether context features improve the fixed Sentinel-2 baseline without changing the model.


In [ ]:
# Locked baseline + context feature experiment.
# Only the feature set changes; model, split, match window, and SS threshold are fixed.

from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
except Exception as exc:
    raise ImportError("XGBoost is required for the locked XGBCls + Ridge baseline.") from exc

LOCKED_MATCH_DAYS = 1
LOCKED_SS_MAX_MG_L = 130.0
LOCKED_LOW_THRESHOLD = 10.0
LOCKED_TRAIN_YEAR_MIN = 2562
LOCKED_TRAIN_YEAR_MAX = 2566
LOCKED_TEST_YEAR_MIN = 2567
LOCKED_TEST_YEAR_MAX = 2568

S2_18_BASELINE_COLS = list(FEATURE_COLS)

CONTEXT_GROUPS = {
    "S2_18_baseline": [],
    "S2_18_plus_rain_runoff_short": [
        "rain_gsmap_prior_24h_mm",
        "rain_gsmap_prior_72h_mm",
        "era5h_runoff_prior_24h_mm",
        "era5h_runoff_prior_72h_mm",
    ],
    "S2_18_plus_hydrology_longer": [
        "rain_gsmap_prior_168h_mm",
        "rain_gsmap_prior_336h_mm",
        "era5h_runoff_prior_168h_mm",
    ],
    "S2_18_plus_water_static_QA": [
        "water_pixel_fraction",
        "candidate_water_count",
        "local_cloud_pct",
        "jrc_occurrence_mean_500m",
        "wc_water_frac_500m",
    ],
}

# A final diagnostic combined set. Interpret only after the smaller groups above.
CONTEXT_GROUPS["S2_18_plus_all_selected_context"] = sorted(set(
    c for group_name, cols in CONTEXT_GROUPS.items() if group_name != "S2_18_baseline" for c in cols
))

all_locked_cols = sorted(set(S2_18_BASELINE_COLS + [c for cols in CONTEXT_GROUPS.values() for c in cols]))
missing_locked_cols = [c for c in all_locked_cols if c not in df.columns]
if missing_locked_cols:
    raise KeyError(f"Missing locked experiment feature columns: {missing_locked_cols}")

locked_year = pd.to_numeric(df["year_be"], errors="coerce")
locked_match_days = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
locked_target = pd.to_numeric(df[TARGET_COL], errors="coerce")

locked_mask = (
    locked_target.notna()
    & locked_match_days.le(LOCKED_MATCH_DAYS)
    & locked_target.le(LOCKED_SS_MAX_MG_L)
)

locked_train_df = df.loc[
    locked_mask & locked_year.between(LOCKED_TRAIN_YEAR_MIN, LOCKED_TRAIN_YEAR_MAX)
].copy()
locked_test_df = df.loc[
    locked_mask & locked_year.between(LOCKED_TEST_YEAR_MIN, LOCKED_TEST_YEAR_MAX)
].copy()

locked_train_df["is_high"] = locked_train_df[TARGET_COL].gt(LOCKED_LOW_THRESHOLD).astype(int)
locked_test_df["is_high"] = locked_test_df[TARGET_COL].gt(LOCKED_LOW_THRESHOLD).astype(int)

locked_low_value = locked_train_df.loc[locked_train_df["is_high"].eq(0), TARGET_COL].median()
locked_high_train_df = locked_train_df.loc[locked_train_df["is_high"].eq(1)].copy()

locked_split_summary = pd.DataFrame([
    {
        "split": "train_2562_2566",
        "rows": len(locked_train_df),
        "stations": locked_train_df["station_canonical"].nunique() if "station_canonical" in locked_train_df.columns else np.nan,
        "target_median": locked_train_df[TARGET_COL].median(),
        "target_p95": locked_train_df[TARGET_COL].quantile(0.95),
        "target_max": locked_train_df[TARGET_COL].max(),
        "low_count": int(locked_train_df["is_high"].eq(0).sum()),
        "high_count": int(locked_train_df["is_high"].eq(1).sum()),
    },
    {
        "split": "test_2567_2568",
        "rows": len(locked_test_df),
        "stations": locked_test_df["station_canonical"].nunique() if "station_canonical" in locked_test_df.columns else np.nan,
        "target_median": locked_test_df[TARGET_COL].median(),
        "target_p95": locked_test_df[TARGET_COL].quantile(0.95),
        "target_max": locked_test_df[TARGET_COL].max(),
        "low_count": int(locked_test_df["is_high"].eq(0).sum()),
        "high_count": int(locked_test_df["is_high"].eq(1).sum()),
    },
])

display(locked_split_summary.round(4))
print("Locked baseline model: XGBCls + Ridge")
print("S2 baseline feature count:", len(S2_18_BASELINE_COLS))
print("Low prediction value:", round(float(locked_low_value), 4))

if locked_train_df.empty or locked_test_df.empty:
    raise ValueError("Locked train/test split is empty. Check match window, years, or SS threshold.")
if locked_train_df["is_high"].nunique() < 2:
    raise ValueError("Locked train split needs both low and high SS classes.")
if locked_high_train_df.empty:
    raise ValueError("No high-SS training samples for Stage 2 regression.")


def locked_make_xgb_classifier(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=5.0,
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
        )),
    ])


def locked_make_ridge_regressor():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ])


def locked_metric_table(y_true_original, y_pred_original):
    y_true_original = np.asarray(y_true_original, dtype=float)
    y_pred_original = np.clip(np.asarray(y_pred_original, dtype=float), 0, None)
    y_true_log = np.log1p(y_true_original)
    y_pred_log = np.log1p(y_pred_original)
    return {
        "original_r2": r2_score(y_true_original, y_pred_original),
        "original_rmse": float(np.sqrt(mean_squared_error(y_true_original, y_pred_original))),
        "original_mae": float(mean_absolute_error(y_true_original, y_pred_original)),
        "log_r2": r2_score(y_true_log, y_pred_log),
        "log_rmse": float(np.sqrt(mean_squared_error(y_true_log, y_pred_log))),
        "spearman_r": safe_corr(spearmanr, y_true_original, y_pred_original),
        "pearson_r": safe_corr(pearsonr, y_true_original, y_pred_original),
        "bias": float(np.mean(y_pred_original - y_true_original)),
    }


def run_locked_xgb_ridge(feature_set_name, context_cols):
    feature_cols = S2_18_BASELINE_COLS + list(context_cols)
    feature_cols = list(dict.fromkeys(feature_cols))

    classifier = locked_make_xgb_classifier(RANDOM_STATE)
    regressor = locked_make_ridge_regressor()

    X_train = locked_train_df[feature_cols]
    X_test = locked_test_df[feature_cols]

    classifier.fit(X_train, locked_train_df["is_high"])
    pred_high = np.asarray(classifier.predict(X_test)).astype(int).ravel()

    regressor.fit(locked_high_train_df[feature_cols], locked_high_train_df[TARGET_LOG_COL])
    pred_high_log = np.asarray(regressor.predict(X_test)).ravel()
    pred_high_original = np.expm1(np.clip(pred_high_log, 0, None))

    pred_original = np.where(pred_high == 1, pred_high_original, locked_low_value)
    metrics = locked_metric_table(locked_test_df[TARGET_COL].values, pred_original)

    tn, fp, fn, tp = confusion_matrix(locked_test_df["is_high"], pred_high, labels=[0, 1]).ravel()

    row = {
        "feature_set": feature_set_name,
        "model": "XGBCls + Ridge",
        "n_features": len(feature_cols),
        "n_s2_features": len(S2_18_BASELINE_COLS),
        "n_context_features": len(context_cols),
        "n_train": len(locked_train_df),
        "n_train_high_only": len(locked_high_train_df),
        "n_test": len(locked_test_df),
        "test_true_high": int(locked_test_df["is_high"].sum()),
        "test_pred_high": int(pred_high.sum()),
        "classifier_accuracy": accuracy_score(locked_test_df["is_high"], pred_high),
        "classifier_balanced_accuracy": balanced_accuracy_score(locked_test_df["is_high"], pred_high),
        "classifier_f1_high": f1_score(locked_test_df["is_high"], pred_high, pos_label=1),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        **metrics,
        "context_features": ", ".join(context_cols) if context_cols else "",
    }

    pred_table = pd.DataFrame({
        "sample_id": locked_test_df["sample_id"].values if "sample_id" in locked_test_df.columns else locked_test_df.index.values,
        "station_canonical": locked_test_df["station_canonical"].values if "station_canonical" in locked_test_df.columns else np.nan,
        "year_be": locked_test_df["year_be"].values,
        "observed_original": locked_test_df[TARGET_COL].values,
        "predicted_original": pred_original,
        "residual_original": pred_original - locked_test_df[TARGET_COL].values,
        "true_high": locked_test_df["is_high"].values,
        "predicted_high": pred_high,
    })
    pred_table["abs_residual_original"] = pred_table["residual_original"].abs()

    return row, pred_table

locked_rows = []
locked_prediction_tables = {}
for feature_set_name, context_cols in CONTEXT_GROUPS.items():
    row, pred_table = run_locked_xgb_ridge(feature_set_name, context_cols)
    locked_rows.append(row)
    locked_prediction_tables[feature_set_name] = pred_table

locked_context_results = pd.DataFrame(locked_rows).sort_values(
    ["original_r2", "log_r2", "original_rmse"],
    ascending=[False, False, True],
).reset_index(drop=True)

# Add deltas against the S2-only locked baseline.
locked_baseline_r2 = float(locked_context_results.loc[
    locked_context_results["feature_set"].eq("S2_18_baseline"), "original_r2"
].iloc[0])
locked_baseline_rmse = float(locked_context_results.loc[
    locked_context_results["feature_set"].eq("S2_18_baseline"), "original_rmse"
].iloc[0])
locked_context_results["delta_r2_vs_s2"] = locked_context_results["original_r2"] - locked_baseline_r2
locked_context_results["delta_rmse_vs_s2"] = locked_context_results["original_rmse"] - locked_baseline_rmse

locked_show_cols = [
    "feature_set", "model", "n_features", "n_context_features",
    "original_r2", "original_rmse", "original_mae",
    "log_r2", "log_rmse", "spearman_r", "bias",
    "delta_r2_vs_s2", "delta_rmse_vs_s2",
    "classifier_accuracy", "classifier_balanced_accuracy", "classifier_f1_high",
    "test_true_high", "test_pred_high", "tn", "fp", "fn", "tp",
]

print("Locked XGBCls + Ridge context experiment results")
display(locked_context_results[locked_show_cols].round(4))

print("Context features used in each set")
display(locked_context_results[["feature_set", "context_features"]])

best_locked_context = locked_context_results.iloc[0]
print("Best locked feature set:", best_locked_context["feature_set"])
print("Best locked original R2:", round(float(best_locked_context["original_r2"]), 4))

best_locked_pred = locked_prediction_tables[best_locked_context["feature_set"]].copy()
print("Largest residuals for the best locked feature set")
display(best_locked_pred.sort_values("abs_residual_original", ascending=False).head(15).round(4))


## Robust Stage-2 Regressors and Classifier Threshold Tuning

This experiment improves the two-stage SS model without tuning directly on the final test set.

Workflow:
1. Keep the same data condition as the direct-threshold sensitivity setup.
2. Split train internally into `train_inner = 2562-2565` and `validation = 2566`.
3. Tune the high/low classifier probability threshold on validation only.
4. Refit the selected classifier/regressor on `2562-2566`.
5. Evaluate once on `2567-2568`.

Robust Stage-2 regressors tested here include `HuberRegressor`, `RANSACRegressor`, `TheilSenRegressor`, and `QuantileRegressor`, plus Ridge/PLS/BayesianRidge references.


In [ ]:
# Robust two-stage tuning config.
# The defaults follow the current direct-threshold SS experiment.

ROBUST_SS_MAX_MATCH_DAYS = DIRECT_SS_MAX_MATCH_DAYS if "DIRECT_SS_MAX_MATCH_DAYS" in globals() else 1
ROBUST_SS_MAX_MG_L = DIRECT_SS_MAX_MG_L if "DIRECT_SS_MAX_MG_L" in globals() else 130.0
ROBUST_SS_LOW_THRESHOLD = LOW_THRESHOLD if "LOW_THRESHOLD" in globals() else 10.0
ROBUST_SS_RANDOM_STATE = RANDOM_STATE
ROBUST_FEATURE_COLS = list(FEATURE_COLS)

ROBUST_TRAIN_INNER_YEAR_MIN = 2562
ROBUST_TRAIN_INNER_YEAR_MAX = 2565
ROBUST_VALIDATION_YEAR = 2566
ROBUST_FINAL_TRAIN_YEAR_MIN = 2562
ROBUST_FINAL_TRAIN_YEAR_MAX = 2566
ROBUST_TEST_YEAR_MIN = 2567
ROBUST_TEST_YEAR_MAX = 2568

# Probability threshold for deciding whether a sample should use the high-SS regressor.
CLASSIFIER_THRESHOLD_GRID = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

print("Robust two-stage tuning config")
print(f"Match window: abs_date_diff_days <= {ROBUST_SS_MAX_MATCH_DAYS}")
print(f"SS threshold: {ROBUST_SS_MAX_MG_L} mg/L")
print(f"Low/high threshold: SS > {ROBUST_SS_LOW_THRESHOLD} mg/L")
print(f"Features: {len(ROBUST_FEATURE_COLS)}")
print(f"Inner train years: {ROBUST_TRAIN_INNER_YEAR_MIN}-{ROBUST_TRAIN_INNER_YEAR_MAX}")
print(f"Validation year: {ROBUST_VALIDATION_YEAR}")
print(f"Final test years: {ROBUST_TEST_YEAR_MIN}-{ROBUST_TEST_YEAR_MAX}")


In [ ]:
# Build robust-tuning train/validation/test splits.

required_robust_cols = ["year_be", "abs_date_diff_days", TARGET_COL, TARGET_LOG_COL] + ROBUST_FEATURE_COLS
missing_robust_cols = [c for c in required_robust_cols if c not in df.columns]
if missing_robust_cols:
    raise KeyError(f"Missing columns for robust SS tuning: {missing_robust_cols}")

year_num = pd.to_numeric(df["year_be"], errors="coerce")
diff_num = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
target_num = pd.to_numeric(df[TARGET_COL], errors="coerce")

robust_mask = target_num.notna() & diff_num.le(ROBUST_SS_MAX_MATCH_DAYS)
if ROBUST_SS_MAX_MG_L is not None:
    robust_mask &= target_num.le(float(ROBUST_SS_MAX_MG_L))

robust_all_df = df.loc[robust_mask].copy()
robust_all_df["is_high"] = robust_all_df[TARGET_COL].gt(ROBUST_SS_LOW_THRESHOLD).astype(int)

robust_train_inner_df = robust_all_df.loc[
    pd.to_numeric(robust_all_df["year_be"], errors="coerce").between(
        ROBUST_TRAIN_INNER_YEAR_MIN, ROBUST_TRAIN_INNER_YEAR_MAX
    )
].copy()
robust_val_df = robust_all_df.loc[
    pd.to_numeric(robust_all_df["year_be"], errors="coerce").eq(ROBUST_VALIDATION_YEAR)
].copy()
robust_final_train_df = robust_all_df.loc[
    pd.to_numeric(robust_all_df["year_be"], errors="coerce").between(
        ROBUST_FINAL_TRAIN_YEAR_MIN, ROBUST_FINAL_TRAIN_YEAR_MAX
    )
].copy()
robust_test_df = robust_all_df.loc[
    pd.to_numeric(robust_all_df["year_be"], errors="coerce").between(
        ROBUST_TEST_YEAR_MIN, ROBUST_TEST_YEAR_MAX
    )
].copy()

robust_split_summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(data),
        "stations": data["station_canonical"].nunique() if "station_canonical" in data.columns else np.nan,
        "target_median": data[TARGET_COL].median(),
        "target_p95": data[TARGET_COL].quantile(0.95),
        "target_max": data[TARGET_COL].max(),
        "low_count": int(data["is_high"].eq(0).sum()) if len(data) else 0,
        "high_count": int(data["is_high"].eq(1).sum()) if len(data) else 0,
    }
    for name, data in [
        ("train_inner_2562_2565", robust_train_inner_df),
        ("validation_2566", robust_val_df),
        ("final_train_2562_2566", robust_final_train_df),
        ("test_2567_2568", robust_test_df),
    ]
])

display(robust_split_summary.round(4))

if robust_val_df.empty:
    raise ValueError("Validation split is empty. Check ROBUST_VALIDATION_YEAR and filters.")
if robust_train_inner_df["is_high"].nunique() < 2:
    raise ValueError("Inner train split needs both low and high SS classes.")
if robust_final_train_df["is_high"].nunique() < 2:
    raise ValueError("Final train split needs both low and high SS classes.")
if robust_train_inner_df["is_high"].sum() < 10:
    print("Warning: inner high-SS train count is small. Robust regressors may be unstable.")
if len(robust_val_df) < 10:
    print("Warning: validation set has fewer than 10 rows. Threshold tuning may be unstable.")


In [ ]:
# Robust models and helper functions.

from sklearn.base import clone
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import HuberRegressor, RANSACRegressor, TheilSenRegressor, QuantileRegressor, LinearRegression


def robust_metrics_from_original(y_true_original, y_pred_original):
    y_true_original = np.asarray(y_true_original, dtype=float)
    y_pred_original = np.asarray(y_pred_original, dtype=float)
    y_pred_original = np.clip(y_pred_original, 0, None)
    y_true_log = np.log1p(y_true_original)
    y_pred_log = np.log1p(y_pred_original)
    return {
        "original_r2": r2_score(y_true_original, y_pred_original),
        "original_rmse": float(np.sqrt(mean_squared_error(y_true_original, y_pred_original))),
        "original_mae": float(mean_absolute_error(y_true_original, y_pred_original)),
        "log_r2": r2_score(y_true_log, y_pred_log),
        "log_rmse": float(np.sqrt(mean_squared_error(y_true_log, y_pred_log))),
        "spearman_r": safe_corr(spearmanr, y_true_original, y_pred_original),
        "pearson_r": safe_corr(pearsonr, y_true_original, y_pred_original),
        "bias": float(np.mean(y_pred_original - y_true_original)),
    }


def make_ransac_ridge(residual_threshold, min_samples, random_state=42):
    base = Ridge(alpha=1.0)
    try:
        return RANSACRegressor(
            estimator=base,
            min_samples=min_samples,
            residual_threshold=residual_threshold,
            max_trials=300,
            random_state=random_state,
        )
    except TypeError:
        return RANSACRegressor(
            base_estimator=base,
            min_samples=min_samples,
            residual_threshold=residual_threshold,
            max_trials=300,
            random_state=random_state,
        )


def make_robust_stage2_regressors(random_state=42):
    models = {
        "Ridge_standard_ref": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "Ridge_robust_ref": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "BayesianRidge_standard_ref": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", BayesianRidge()),
        ]),
        "PLS_2_standard_ref": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=2)),
        ]),
        "PLS_3_standard_ref": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=3)),
        ]),
    }

    for n_components in [5, 8]:
        models[f"PLS_{n_components}_standard_ref"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=n_components)),
        ])

    for c in [1.0, 3.0, 10.0]:
        for epsilon in [0.05, 0.10, 0.20]:
            models[f"SVR_RBF_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", SVR(kernel="rbf", C=c, epsilon=epsilon, gamma="scale")),
            ])

    for c in [0.5, 1.0, 2.0]:
        for epsilon in [0.0, 0.05]:
            models[f"LinearSVR_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", LinearSVR(C=c, epsilon=epsilon, random_state=random_state, max_iter=30000)),
            ])

    if HAS_CUBIST:
        models["Cubist_c1_ref"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=1, random_state=random_state)),
        ])
        models["Cubist_c5_ref"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Cubist(n_committees=5, random_state=random_state)),
        ])

    for epsilon in [1.10, 1.35, 1.50, 2.00]:
        for alpha in [0.0001, 0.001, 0.01]:
            models[f"Huber_eps{epsilon}_a{alpha}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", HuberRegressor(epsilon=epsilon, alpha=alpha, max_iter=2000)),
            ])

    for residual_threshold in [0.4, 0.6, 0.8, 1.0]:
        for min_samples in [0.7, 0.9]:
            models[f"RANSAC_Ridge_thr{residual_threshold}_ms{min_samples}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", make_ransac_ridge(residual_threshold, min_samples, random_state=random_state)),
            ])

    for max_subpopulation in [1000, 5000]:
        models[f"TheilSen_maxsub{max_subpopulation}"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            ("model", TheilSenRegressor(
                max_subpopulation=max_subpopulation,
                n_subsamples=None,
                random_state=random_state,
                n_jobs=-1,
            )),
        ])

    for quantile in [0.40, 0.50, 0.60]:
        for alpha in [0.0001, 0.001, 0.01]:
            models[f"Quantile_q{quantile}_a{alpha}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", QuantileRegressor(quantile=quantile, alpha=alpha, solver="highs")),
            ])

    return models


def make_threshold_classifiers(random_state=42):
    classifiers = {
        "LogisticBalanced": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=5000, random_state=random_state)),
        ]),
        "SVC_RBF_balanced_proba": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=1.0, gamma="scale", class_weight="balanced", probability=True, random_state=random_state)),
        ]),
        "RandomForestCls": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=600, min_samples_leaf=2, class_weight="balanced", random_state=random_state, n_jobs=-1)),
        ]),
        "ExtraTreesCls": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2, class_weight="balanced", random_state=random_state, n_jobs=-1)),
        ]),
    }
    if HAS_XGBOOST and XGBClassifier is not None:
        classifiers["XGBCls"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=300, max_depth=2, learning_rate=0.03,
                subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
                eval_metric="logloss", random_state=random_state, n_jobs=-1,
            )),
        ])
    if HAS_CATBOOST and CatBoostClassifier is not None:
        classifiers["CatBoostCls"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostClassifier(
                iterations=500, depth=3, learning_rate=0.03,
                loss_function="Logloss", auto_class_weights="Balanced",
                random_seed=random_state, verbose=False, allow_writing_files=False,
            )),
        ])
    return classifiers


def high_probability(classifier, X):
    if hasattr(classifier, "predict_proba"):
        proba = np.asarray(classifier.predict_proba(X))
        if proba.ndim == 2 and proba.shape[1] > 1:
            return proba[:, 1]
        return proba.ravel()
    if hasattr(classifier, "decision_function"):
        score = np.asarray(classifier.decision_function(X), dtype=float).ravel()
        return 1.0 / (1.0 + np.exp(-score))
    return np.asarray(classifier.predict(X), dtype=float).ravel()


def two_stage_predictions_from_proba(proba_high, regressor_pred_original, low_value, threshold):
    pred_high = np.asarray(proba_high) >= threshold
    combined = np.where(pred_high, regressor_pred_original, low_value)
    return pred_high.astype(int), combined


In [ ]:
# Tune classifier threshold and robust Stage-2 regressors on validation year 2566 only.

X_inner = robust_train_inner_df[ROBUST_FEATURE_COLS]
y_inner_class = robust_train_inner_df["is_high"]
X_val = robust_val_df[ROBUST_FEATURE_COLS]
y_val_original = robust_val_df[TARGET_COL].values

inner_low_value = robust_train_inner_df.loc[robust_train_inner_df["is_high"].eq(0), TARGET_COL].median()
inner_high_train = robust_train_inner_df.loc[robust_train_inner_df["is_high"].eq(1)].copy()

threshold_classifiers = make_threshold_classifiers(ROBUST_SS_RANDOM_STATE)
robust_stage2_regressors = make_robust_stage2_regressors(ROBUST_SS_RANDOM_STATE)

fitted_inner_classifiers = {}
val_proba_by_classifier = {}
classifier_fit_errors = []
for classifier_name, classifier in threshold_classifiers.items():
    try:
        model = clone(classifier)
        model.fit(X_inner, y_inner_class)
        fitted_inner_classifiers[classifier_name] = model
        val_proba_by_classifier[classifier_name] = high_probability(model, X_val)
    except Exception as exc:
        classifier_fit_errors.append({"classifier": classifier_name, "error": repr(exc)})

fitted_inner_regressors = {}
val_reg_pred_by_regressor = {}
regressor_fit_errors = []
for regressor_name, regressor in robust_stage2_regressors.items():
    try:
        model = clone(regressor)
        model.fit(inner_high_train[ROBUST_FEATURE_COLS], inner_high_train[TARGET_LOG_COL])
        pred_log = np.asarray(model.predict(X_val)).ravel()
        pred_original = np.expm1(np.clip(pred_log, 0, None))
        fitted_inner_regressors[regressor_name] = model
        val_reg_pred_by_regressor[regressor_name] = pred_original
    except Exception as exc:
        regressor_fit_errors.append({"regressor": regressor_name, "error": repr(exc)})

val_rows = []
for classifier_name, proba_high in val_proba_by_classifier.items():
    for regressor_name, reg_pred_original in val_reg_pred_by_regressor.items():
        for threshold in CLASSIFIER_THRESHOLD_GRID:
            pred_high, combined_pred = two_stage_predictions_from_proba(
                proba_high, reg_pred_original, inner_low_value, threshold
            )
            metrics = robust_metrics_from_original(y_val_original, combined_pred)
            val_rows.append({
                "classifier": classifier_name,
                "regressor": regressor_name,
                "classifier_threshold": threshold,
                "n_train_inner": len(robust_train_inner_df),
                "n_train_inner_high": len(inner_high_train),
                "n_val": len(robust_val_df),
                "val_true_high": int(robust_val_df["is_high"].sum()),
                "val_pred_high": int(pred_high.sum()),
                **{f"val_{k}": v for k, v in metrics.items()},
            })

robust_threshold_val_results = pd.DataFrame(val_rows)
robust_threshold_val_results = robust_threshold_val_results.sort_values(
    ["val_original_r2", "val_log_r2", "val_original_rmse"],
    ascending=[False, False, True],
).reset_index(drop=True)

print("Validation tuning results sorted by validation original-scale R2")
val_show_cols = [
    "classifier", "regressor", "classifier_threshold",
    "n_train_inner", "n_train_inner_high", "n_val", "val_true_high", "val_pred_high",
    "val_original_r2", "val_original_rmse", "val_original_mae", "val_log_r2", "val_log_rmse",
    "val_spearman_r", "val_bias",
]
display(robust_threshold_val_results[val_show_cols].head(30).round(4))

if classifier_fit_errors:
    print("Classifier fit errors")
    display(pd.DataFrame(classifier_fit_errors))
if regressor_fit_errors:
    print("Regressor fit errors")
    display(pd.DataFrame(regressor_fit_errors))

# Best threshold per classifier+regressor combo from validation only.
robust_val_best_by_combo = (
    robust_threshold_val_results
    .sort_values(["classifier", "regressor", "val_original_r2", "val_log_r2"], ascending=[True, True, False, False])
    .groupby(["classifier", "regressor"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# The single config selected only by validation. This is the clean final-candidate config.
robust_selected_by_validation = robust_threshold_val_results.iloc[0].copy()
print("Single best config selected by validation only")
display(pd.DataFrame([robust_selected_by_validation[val_show_cols]]).round(4))


In [ ]:
# Refit validation-tuned configs on 2562-2566 and evaluate on 2567-2568.

X_final_train = robust_final_train_df[ROBUST_FEATURE_COLS]
y_final_class = robust_final_train_df["is_high"]
X_test_robust = robust_test_df[ROBUST_FEATURE_COLS]
y_test_robust_original = robust_test_df[TARGET_COL].values

final_low_value = robust_final_train_df.loc[robust_final_train_df["is_high"].eq(0), TARGET_COL].median()
final_high_train = robust_final_train_df.loc[robust_final_train_df["is_high"].eq(1)].copy()

# Fit each selected classifier/regressor once.
selected_classifier_names = sorted(robust_val_best_by_combo["classifier"].unique())
selected_regressor_names = sorted(robust_val_best_by_combo["regressor"].unique())

final_classifiers = {}
final_classifier_test_proba = {}
final_fit_errors = []
for classifier_name in selected_classifier_names:
    try:
        model = clone(threshold_classifiers[classifier_name])
        model.fit(X_final_train, y_final_class)
        final_classifiers[classifier_name] = model
        final_classifier_test_proba[classifier_name] = high_probability(model, X_test_robust)
    except Exception as exc:
        final_fit_errors.append({"stage": "classifier", "model": classifier_name, "error": repr(exc)})

final_regressors = {}
final_regressor_test_pred = {}
for regressor_name in selected_regressor_names:
    try:
        model = clone(robust_stage2_regressors[regressor_name])
        model.fit(final_high_train[ROBUST_FEATURE_COLS], final_high_train[TARGET_LOG_COL])
        pred_log = np.asarray(model.predict(X_test_robust)).ravel()
        pred_original = np.expm1(np.clip(pred_log, 0, None))
        final_regressors[regressor_name] = model
        final_regressor_test_pred[regressor_name] = pred_original
    except Exception as exc:
        final_fit_errors.append({"stage": "regressor", "model": regressor_name, "error": repr(exc)})

test_rows = []
robust_test_prediction_tables = {}
for _, cfg in robust_val_best_by_combo.iterrows():
    classifier_name = cfg["classifier"]
    regressor_name = cfg["regressor"]
    threshold = float(cfg["classifier_threshold"])
    if classifier_name not in final_classifier_test_proba or regressor_name not in final_regressor_test_pred:
        continue

    pred_high, combined_pred = two_stage_predictions_from_proba(
        final_classifier_test_proba[classifier_name],
        final_regressor_test_pred[regressor_name],
        final_low_value,
        threshold,
    )
    metrics = robust_metrics_from_original(y_test_robust_original, combined_pred)
    row = {
        "classifier": classifier_name,
        "regressor": regressor_name,
        "classifier_threshold": threshold,
        "n_final_train": len(robust_final_train_df),
        "n_final_train_high": len(final_high_train),
        "n_test": len(robust_test_df),
        "test_true_high": int(robust_test_df["is_high"].sum()),
        "test_pred_high": int(pred_high.sum()),
        "selected_val_original_r2": cfg["val_original_r2"],
        "selected_val_original_rmse": cfg["val_original_rmse"],
        "selected_val_log_r2": cfg["val_log_r2"],
        **{f"test_{k}": v for k, v in metrics.items()},
    }
    test_rows.append(row)

    robust_test_prediction_tables[(classifier_name, regressor_name, threshold)] = pd.DataFrame({
        "sample_id": robust_test_df["sample_id"].values if "sample_id" in robust_test_df.columns else robust_test_df.index.values,
        "station_canonical": robust_test_df["station_canonical"].values if "station_canonical" in robust_test_df.columns else np.nan,
        "year_be": robust_test_df["year_be"].values,
        "observed_original": y_test_robust_original,
        "predicted_original": combined_pred,
        "residual_original": combined_pred - y_test_robust_original,
        "predicted_high": pred_high,
        "true_high": robust_test_df["is_high"].values,
        "proba_high": final_classifier_test_proba[classifier_name],
    })

robust_threshold_test_results = pd.DataFrame(test_rows)
robust_threshold_test_results = robust_threshold_test_results.sort_values(
    ["test_original_r2", "test_log_r2", "test_original_rmse"],
    ascending=[False, False, True],
).reset_index(drop=True)

# Mark the single validation-selected config in the test table.
selected_key = (
    robust_selected_by_validation["classifier"],
    robust_selected_by_validation["regressor"],
    float(robust_selected_by_validation["classifier_threshold"]),
)
robust_threshold_test_results["is_single_best_by_validation"] = (
    robust_threshold_test_results["classifier"].eq(selected_key[0])
    & robust_threshold_test_results["regressor"].eq(selected_key[1])
    & robust_threshold_test_results["classifier_threshold"].eq(selected_key[2])
)

test_show_cols = [
    "is_single_best_by_validation", "classifier", "regressor", "classifier_threshold",
    "n_final_train", "n_final_train_high", "n_test", "test_true_high", "test_pred_high",
    "selected_val_original_r2", "test_original_r2", "test_original_rmse", "test_original_mae",
    "test_log_r2", "test_log_rmse", "test_spearman_r", "test_bias",
]

print("Final test result for the single config selected by validation only")
display(
    robust_threshold_test_results.loc[
        robust_threshold_test_results["is_single_best_by_validation"], test_show_cols
    ].round(4)
)

print("All validation-tuned configs evaluated on final test, sorted by test R2 (diagnostic ranking)")
display(robust_threshold_test_results[test_show_cols].head(30).round(4))

if final_fit_errors:
    print("Final fit errors")
    display(pd.DataFrame(final_fit_errors))


In [ ]:
# Plot and inspect the best robust-threshold result on final test.
# This plot uses the best test-ranked config for diagnosis; the validation-selected row above is the cleaner final candidate.

if robust_threshold_test_results.empty:
    print("No robust threshold test results available.")
else:
    best_diag = robust_threshold_test_results.iloc[0]
    best_diag_key = (
        best_diag["classifier"],
        best_diag["regressor"],
        float(best_diag["classifier_threshold"]),
    )
    best_diag_pred = robust_test_prediction_tables[best_diag_key].copy()
    best_diag_pred["abs_residual_original"] = best_diag_pred["residual_original"].abs()

    print("Best diagnostic test-ranked robust config:")
    display(pd.DataFrame([best_diag[test_show_cols]]).round(4))

    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(best_diag_pred["observed_original"], best_diag_pred["predicted_original"], alpha=0.8)
    limit = max(best_diag_pred["observed_original"].max(), best_diag_pred["predicted_original"].max()) * 1.05
    ax.plot([0, limit], [0, limit], linestyle="--", color="gray", linewidth=1)
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_xlabel("Observed SS (mg/L)")
    ax.set_ylabel("Predicted SS (mg/L)")
    ax.set_title(f"Robust threshold tuning | R2={best_diag['test_original_r2']:.3f}")
    ax.grid(True, alpha=0.3)
    plt.show()

    print("Largest residuals for best diagnostic test-ranked robust config")
    display(best_diag_pred.sort_values("abs_residual_original", ascending=False).head(20).round(4))


## Cross-Validation Split: SS

This section checks whether the SS model performance is stable under cross-validation.

It uses the same default data condition as the direct-threshold sensitivity experiment:
- `abs_date_diff_days <= 1`
- `SS <= 150 mg/L`

Two split modes are reported:
- `random_kfold`: sample-level shuffled KFold. This is useful as a diagnostic but can be optimistic when nearby/repeated station samples are split across folds.
- `station_group_kfold`: holds out stations by fold using `station_canonical`. This is stricter for station leakage, but the folds can be harder and less balanced.

This does not replace the temporal holdout. Use it as stability evidence alongside the 2567-2568 test result.


In [ ]:
# Cross-validation config for SS.
# Change these values and rerun this section only.

CV_SS_MAX_MATCH_DAYS = 1
CV_SS_MAX_MG_L = 130.0          # Set to None to disable direct threshold filtering.
CV_SS_N_SPLITS = 5
CV_SS_RANDOM_STATE = RANDOM_STATE
CV_SS_LOW_THRESHOLD = LOW_THRESHOLD if "LOW_THRESHOLD" in globals() else 10.0
CV_SS_FEATURE_COLS = list(FEATURE_COLS)
CV_SS_SPLIT_MODES = ["random_kfold", "station_group_kfold"]

# Keep the CV model list focused so the notebook stays runnable.
CV_SS_ONE_STAGE_MODELS = [
    "PLS_3", "BayesianRidge", "Cubist_c1", "Cubist_c5", "Cubist_c5_n3",
    "CatBoost", "RandomForest", "XGBoost", "Ridge", "ElasticNet"
]
CV_SS_TWO_STAGE_CLASSIFIERS = [
    "CatBoostCls", "SVC_RBF_balanced", "LogisticBalanced", "RandomForestCls"
]
CV_SS_TWO_STAGE_REGRESSORS = [
    "Ridge", "ElasticNet", "PLS_2", "BayesianRidge",
    "Cubist_c1", "Cubist_c5", "Cubist_c5_n3"
]

if not CV_SS_FEATURE_COLS:
    raise ValueError("CV_SS_FEATURE_COLS is empty. Fill FEATURE_COLS first, then rerun this section.")

print("SS cross-validation config")
print(f"Match window: abs_date_diff_days <= {CV_SS_MAX_MATCH_DAYS}")
print(f"SS threshold: {CV_SS_MAX_MG_L} mg/L")
print(f"CV folds: {CV_SS_N_SPLITS}")
print(f"Split modes: {CV_SS_SPLIT_MODES}")
print(f"Feature count: {len(CV_SS_FEATURE_COLS)}")


In [ ]:
# Build the SS cross-validation dataset.

from sklearn.base import clone
from sklearn.model_selection import KFold, GroupKFold

required_cv_cols = ["year_be", "abs_date_diff_days", TARGET_COL, TARGET_LOG_COL] + CV_SS_FEATURE_COLS
missing_cv_cols = [c for c in required_cv_cols if c not in df.columns]
if missing_cv_cols:
    raise KeyError(f"Missing columns for SS cross-validation: {missing_cv_cols}")

if TARGET_LOG_COL not in df.columns:
    df[TARGET_LOG_COL] = np.log1p(pd.to_numeric(df[TARGET_COL], errors="coerce"))

year_num = pd.to_numeric(df["year_be"], errors="coerce")
diff_num = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
target_num = pd.to_numeric(df[TARGET_COL], errors="coerce")

cv_mask = target_num.notna() & diff_num.le(CV_SS_MAX_MATCH_DAYS)
cv_mask &= year_num.between(TRAIN_YEAR_MIN, TEST_YEAR_MAX)
if CV_SS_MAX_MG_L is not None:
    cv_mask &= target_num.le(float(CV_SS_MAX_MG_L))

ss_cv_df = df.loc[cv_mask].copy().reset_index(drop=True)
ss_cv_df["is_high"] = ss_cv_df[TARGET_COL].gt(CV_SS_LOW_THRESHOLD).astype(int)

cv_data_summary = pd.DataFrame({
    "condition": ["ss_cv_dataset"],
    "rows": [len(ss_cv_df)],
    "stations": [ss_cv_df["station_canonical"].nunique() if "station_canonical" in ss_cv_df.columns else np.nan],
    "years": [f"{int(ss_cv_df['year_be'].min())}-{int(ss_cv_df['year_be'].max())}" if len(ss_cv_df) else ""],
    "target_median": [ss_cv_df[TARGET_COL].median()],
    "target_p95": [ss_cv_df[TARGET_COL].quantile(0.95)],
    "target_max": [ss_cv_df[TARGET_COL].max()],
    "low_or_censored_count": [int(ss_cv_df["is_high"].eq(0).sum())],
    "high_count": [int(ss_cv_df["is_high"].eq(1).sum())],
})

display(cv_data_summary.round(4))
if len(ss_cv_df) < CV_SS_N_SPLITS * 5:
    print("Warning: CV dataset is small. Fold metrics may be unstable.")

# Create folds.
cv_splits = []
if "random_kfold" in CV_SS_SPLIT_MODES:
    kfold = KFold(n_splits=CV_SS_N_SPLITS, shuffle=True, random_state=CV_SS_RANDOM_STATE)
    for fold, (train_idx, test_idx) in enumerate(kfold.split(ss_cv_df), start=1):
        cv_splits.append(("random_kfold", fold, train_idx, test_idx))

if "station_group_kfold" in CV_SS_SPLIT_MODES:
    if "station_canonical" not in ss_cv_df.columns:
        print("station_group_kfold skipped because station_canonical is missing.")
    else:
        groups = ss_cv_df["station_canonical"].astype(str).values
        n_groups = pd.Series(groups).nunique()
        n_splits = min(CV_SS_N_SPLITS, n_groups)
        if n_splits < 2:
            print("station_group_kfold skipped because fewer than 2 station groups are available.")
        else:
            group_kfold = GroupKFold(n_splits=n_splits)
            for fold, (train_idx, test_idx) in enumerate(group_kfold.split(ss_cv_df, groups=groups), start=1):
                cv_splits.append(("station_group_kfold", fold, train_idx, test_idx))

fold_rows = []
for split_mode, fold, train_idx, test_idx in cv_splits:
    train_part = ss_cv_df.iloc[train_idx]
    test_part = ss_cv_df.iloc[test_idx]
    fold_rows.append({
        "split_mode": split_mode,
        "fold": fold,
        "n_train": len(train_part),
        "n_test": len(test_part),
        "train_stations": train_part["station_canonical"].nunique() if "station_canonical" in ss_cv_df.columns else np.nan,
        "test_stations": test_part["station_canonical"].nunique() if "station_canonical" in ss_cv_df.columns else np.nan,
        "train_target_max": train_part[TARGET_COL].max(),
        "test_target_max": test_part[TARGET_COL].max(),
        "train_high": int(train_part["is_high"].sum()),
        "test_high": int(test_part["is_high"].sum()),
    })

ss_cv_fold_summary = pd.DataFrame(fold_rows)
display(ss_cv_fold_summary.round(4))


In [ ]:
# Run SS cross-validation for one-stage and two-stage models.

def cv_metrics_from_original_with_log(y_true_original, y_pred_original):
    y_true_original = np.asarray(y_true_original, dtype=float)
    y_pred_original = np.asarray(y_pred_original, dtype=float)
    y_pred_original = np.clip(y_pred_original, 0, None)
    out = metrics_from_original(y_true_original, y_pred_original)
    y_true_log = np.log1p(y_true_original)
    y_pred_log = np.log1p(y_pred_original)
    out["log_r2"] = r2_score(y_true_log, y_pred_log)
    out["log_rmse"] = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    return out

one_stage_model_pool = make_models(CV_SS_RANDOM_STATE)
one_stage_models = {
    name: model for name, model in one_stage_model_pool.items()
    if name in CV_SS_ONE_STAGE_MODELS
}
missing_one_stage_models = [name for name in CV_SS_ONE_STAGE_MODELS if name not in one_stage_models]
if missing_one_stage_models:
    print("Skipped unavailable one-stage models:", missing_one_stage_models)

classifier_pool = make_classifiers(CV_SS_RANDOM_STATE)
classifier_models = {
    name: model for name, model in classifier_pool.items()
    if name in CV_SS_TWO_STAGE_CLASSIFIERS
}
missing_classifiers = [name for name in CV_SS_TWO_STAGE_CLASSIFIERS if name not in classifier_models]
if missing_classifiers:
    print("Skipped unavailable classifiers:", missing_classifiers)

regressor_pool = make_high_regressors(CV_SS_RANDOM_STATE)
regressor_models = {
    name: model for name, model in regressor_pool.items()
    if name in CV_SS_TWO_STAGE_REGRESSORS
}
missing_regressors = [name for name in CV_SS_TWO_STAGE_REGRESSORS if name not in regressor_models]
if missing_regressors:
    print("Skipped unavailable high-SS regressors:", missing_regressors)

cv_rows = []
cv_pred_rows = []

for split_mode, fold, train_idx, test_idx in cv_splits:
    train_fold = ss_cv_df.iloc[train_idx].copy()
    test_fold = ss_cv_df.iloc[test_idx].copy()

    X_train = train_fold[CV_SS_FEATURE_COLS]
    X_test = test_fold[CV_SS_FEATURE_COLS]
    y_train_log = train_fold[TARGET_LOG_COL]
    y_test_log = test_fold[TARGET_LOG_COL]
    y_test_original = test_fold[TARGET_COL].values

    # One-stage log-regression.
    for model_name, model_template in one_stage_models.items():
        model = clone(model_template)
        model.fit(X_train, y_train_log)
        pred_log = np.asarray(model.predict(X_test)).ravel()
        pred_original = np.expm1(np.clip(pred_log, 0, None))
        metrics = regression_metrics(y_test_log, pred_log, y_test_original)
        row = {
            "approach": "one_stage_log_regression",
            "split_mode": split_mode,
            "fold": fold,
            "model": model_name,
            "n_features": len(CV_SS_FEATURE_COLS),
            "n_train": len(train_fold),
            "n_test": len(test_fold),
            "train_high": int(train_fold["is_high"].sum()),
            "test_high": int(test_fold["is_high"].sum()),
        }
        row.update(metrics)
        cv_rows.append(row)
        cv_pred_rows.extend([
            {
                "approach": "one_stage_log_regression",
                "split_mode": split_mode,
                "fold": fold,
                "model": model_name,
                "sample_id": sample_id,
                "station_canonical": station,
                "observed_original": obs,
                "predicted_original": pred,
            }
            for sample_id, station, obs, pred in zip(
                test_fold["sample_id"].values if "sample_id" in test_fold.columns else test_fold.index.values,
                test_fold["station_canonical"].values if "station_canonical" in test_fold.columns else [np.nan] * len(test_fold),
                y_test_original,
                pred_original,
            )
        ])

    # Two-stage low/high model. Low/high decisions and low-value are learned inside each fold.
    if train_fold["is_high"].nunique() < 2 or train_fold["is_high"].sum() < 10:
        continue

    low_value_fold = train_fold.loc[train_fold["is_high"].eq(0), TARGET_COL].median()
    high_train_fold = train_fold.loc[train_fold["is_high"].eq(1)].copy()

    classifier_predictions = {}
    for classifier_name, classifier_template in classifier_models.items():
        classifier = clone(classifier_template)
        classifier.fit(X_train, train_fold["is_high"])
        pred_high = np.asarray(classifier.predict(X_test)).ravel().astype(int)
        classifier_predictions[classifier_name] = pred_high

    regressor_predictions = {}
    for regressor_name, regressor_template in regressor_models.items():
        regressor = clone(regressor_template)
        regressor.fit(high_train_fold[CV_SS_FEATURE_COLS], high_train_fold[TARGET_LOG_COL])
        pred_log = np.asarray(regressor.predict(X_test)).ravel()
        regressor_predictions[regressor_name] = np.expm1(np.clip(pred_log, 0, None))

    for classifier_name, pred_high in classifier_predictions.items():
        for regressor_name, reg_pred_original in regressor_predictions.items():
            combined_pred = np.where(pred_high == 1, reg_pred_original, low_value_fold)
            metrics = cv_metrics_from_original_with_log(y_test_original, combined_pred)
            model_name = f"{classifier_name} + {regressor_name}"
            row = {
                "approach": "two_stage_low_high",
                "split_mode": split_mode,
                "fold": fold,
                "model": model_name,
                "n_features": len(CV_SS_FEATURE_COLS),
                "n_train": len(train_fold),
                "n_test": len(test_fold),
                "train_high": int(train_fold["is_high"].sum()),
                "test_high": int(test_fold["is_high"].sum()),
                "pred_high": int(pred_high.sum()),
            }
            row.update(metrics)
            cv_rows.append(row)
            cv_pred_rows.extend([
                {
                    "approach": "two_stage_low_high",
                    "split_mode": split_mode,
                    "fold": fold,
                    "model": model_name,
                    "sample_id": sample_id,
                    "station_canonical": station,
                    "observed_original": obs,
                    "predicted_original": pred,
                    "predicted_high": high_flag,
                    "true_high": true_flag,
                }
                for sample_id, station, obs, pred, high_flag, true_flag in zip(
                    test_fold["sample_id"].values if "sample_id" in test_fold.columns else test_fold.index.values,
                    test_fold["station_canonical"].values if "station_canonical" in test_fold.columns else [np.nan] * len(test_fold),
                    y_test_original,
                    combined_pred,
                    pred_high,
                    test_fold["is_high"].values,
                )
            ])

ss_cv_fold_results = pd.DataFrame(cv_rows)
ss_cv_predictions = pd.DataFrame(cv_pred_rows)

print("SS CV fold-level results rows:", len(ss_cv_fold_results))
display(ss_cv_fold_results.head().round(4))


In [ ]:
# Summarize SS cross-validation metrics.

metric_cols = ["original_r2", "original_rmse", "original_mae", "log_r2", "log_rmse", "spearman_r", "pearson_r", "bias"]
summary_parts = []

for keys, group in ss_cv_fold_results.groupby(["split_mode", "approach", "model"], dropna=False):
    split_mode, approach, model = keys
    row = {
        "split_mode": split_mode,
        "approach": approach,
        "model": model,
        "n_folds": group["fold"].nunique(),
        "n_train_mean": group["n_train"].mean(),
        "n_test_mean": group["n_test"].mean(),
    }
    for col in metric_cols:
        row[f"{col}_mean"] = group[col].mean()
        row[f"{col}_std"] = group[col].std(ddof=1)
    summary_parts.append(row)

ss_cv_summary = pd.DataFrame(summary_parts).sort_values(
    ["split_mode", "original_r2_mean", "log_r2_mean"],
    ascending=[True, False, False],
).reset_index(drop=True)

ss_cv_show_cols = [
    "split_mode", "approach", "model", "n_folds", "n_train_mean", "n_test_mean",
    "original_r2_mean", "original_r2_std", "original_rmse_mean", "original_rmse_std",
    "log_r2_mean", "log_r2_std", "log_rmse_mean", "log_rmse_std",
    "spearman_r_mean", "spearman_r_std", "bias_mean",
]

print("Best SS cross-validation models by split mode")
best_by_split = ss_cv_summary.sort_values(
    ["split_mode", "original_r2_mean", "log_r2_mean"],
    ascending=[True, False, False],
).groupby("split_mode", as_index=False).head(10)
display(best_by_split[ss_cv_show_cols].round(4))

print("Best SS cross-validation model within each approach")
best_by_approach = ss_cv_summary.sort_values(
    ["split_mode", "approach", "original_r2_mean", "log_r2_mean"],
    ascending=[True, True, False, False],
).groupby(["split_mode", "approach"], as_index=False).head(1)
display(best_by_approach[ss_cv_show_cols].round(4))

# Simple plot: top models by original R2 mean for each split mode.
for split_mode in ss_cv_summary["split_mode"].drop_duplicates():
    plot_df = ss_cv_summary.loc[ss_cv_summary["split_mode"].eq(split_mode)].head(12).copy()
    labels = plot_df["approach"].str.replace("_", " ") + " | " + plot_df["model"]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(plot_df))))
    ax.barh(labels[::-1], plot_df["original_r2_mean"][::-1])
    ax.set_xlabel("Mean original-scale R2")
    ax.set_title(f"SS CV performance: {split_mode}")
    ax.axvline(0, color="gray", linewidth=1)
    plt.tight_layout()
    plt.show()


## SHAP Explainability for the Best SS Two-Stage Model

The best reportable SS model in this notebook is a two-stage model, not a single regression model:

1. Stage 1 classifier: predict whether `SS > 10 mg/L` or `SS <= 10 mg/L`.
2. Stage 2 high-SS regressor: if the classifier predicts high SS, estimate `log1p(SS)` and convert back to mg/L. If the classifier predicts low/censored SS, the model returns the train-set low median value.

Therefore, SHAP is reported separately for the classifier and the high-SS regressor. The classifier SHAP explains why a sample is classified as high SS, while the regressor SHAP explains the predicted SS value among samples treated as high SS.


In [ ]:
# Refit the best actual two-stage SS model and prepare SHAP inputs.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm.auto")

try:
    import shap
except ImportError as exc:
    raise ImportError(
        "SHAP is not installed in this kernel. Install it with: python -m pip install shap"
    ) from exc

# Pick the best deployable two-stage model, excluding the oracle diagnostic upper bound.
best_two_stage_actual = (
    two_stage_results.loc[~two_stage_results["classifier"].astype(str).str.startswith("ORACLE")]
    .sort_values(["original_r2", "spearman_r"], ascending=False)
    .iloc[0]
)
BEST_SS_CLASSIFIER_NAME = best_two_stage_actual["classifier"]
BEST_SS_REGRESSOR_NAME = best_two_stage_actual["regressor"]

best_classifier = make_classifiers(RANDOM_STATE)[BEST_SS_CLASSIFIER_NAME]
best_regressor = make_high_regressors(RANDOM_STATE)[BEST_SS_REGRESSOR_NAME]

best_classifier.fit(train_stage[FEATURE_COLS], train_stage["is_high"])
best_high_pred = np.asarray(best_classifier.predict(test_stage[FEATURE_COLS])).ravel().astype(int)

best_regressor.fit(high_train[FEATURE_COLS], high_train[TARGET_LOG_COL])
best_reg_pred_log = np.asarray(best_regressor.predict(test_stage[FEATURE_COLS])).ravel()
best_reg_pred_original = np.expm1(np.clip(best_reg_pred_log, 0, None))
best_combined_pred = np.where(best_high_pred == 1, best_reg_pred_original, low_value)

def two_stage_metrics_with_log(y_true_original, y_pred_original):
    y_true_original = np.asarray(y_true_original, dtype=float)
    y_pred_original = np.asarray(y_pred_original, dtype=float)
    y_true_log = np.log1p(y_true_original)
    y_pred_log = np.log1p(np.clip(y_pred_original, 0, None))
    return {
        "original_r2": r2_score(y_true_original, y_pred_original),
        "original_rmse": np.sqrt(mean_squared_error(y_true_original, y_pred_original)),
        "original_mae": mean_absolute_error(y_true_original, y_pred_original),
        "log_r2": r2_score(y_true_log, y_pred_log),
        "log_rmse": np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        "spearman_r": safe_corr(spearmanr, y_true_original, y_pred_original),
        "pearson_r": safe_corr(pearsonr, y_true_original, y_pred_original),
        "bias": float(np.mean(y_pred_original - y_true_original)),
    }

best_two_stage_metrics = two_stage_metrics_with_log(test_stage[TARGET_COL].values, best_combined_pred)

best_two_stage_summary = pd.DataFrame([{
    "target": "SS",
    "classifier": BEST_SS_CLASSIFIER_NAME,
    "regressor": BEST_SS_REGRESSOR_NAME,
    "low_threshold": LOW_THRESHOLD,
    "low_prediction_value": low_value,
    "n_train_classifier": len(train_stage),
    "n_train_high_regressor": len(high_train),
    "n_test": len(test_stage),
    "pred_high": int(best_high_pred.sum()),
    "true_high": int(test_stage["is_high"].sum()),
    **best_two_stage_metrics,
}])

display(best_two_stage_summary.round(4))


### Coefficient Interpretation for the SS Two-Stage Model

The best deployable SS model is `XGBCls + Ridge`. The classifier (`XGBCls`) is a tree model, so it does not have linear coefficients. It should be interpreted with SHAP or tree feature importance.

The second-stage regressor (`Ridge`) is a linear model, so its standardized coefficients can be reported. These coefficients explain the high-SS regression step only, i.e. the part that estimates `log1p(SS)` for samples classified as `SS > 10 mg/L`.


In [ ]:
# Coefficients for the Stage 2 high-SS Ridge regressor.
# These coefficients are for median-imputed + standardized Sentinel-2 features.
# Target scale: log1p(SS), trained only on high-SS samples (SS > 10 mg/L).

stage2_regressor_preprocess = Pipeline(best_regressor.steps[:-1])
stage2_regressor_final_model = best_regressor.steps[-1][1]

if not hasattr(stage2_regressor_final_model, "coef_"):
    print(f"{BEST_SS_REGRESSOR_NAME} does not expose linear coefficients. Use SHAP/feature importance instead.")
else:
    stage2_coef = np.asarray(stage2_regressor_final_model.coef_, dtype=float).ravel()
    stage2_intercept = float(np.asarray(stage2_regressor_final_model.intercept_).ravel()[0])

    stage2_coef_table = (
        pd.DataFrame({
            "feature": FEATURE_COLS,
            "coefficient_log1p_standardized": stage2_coef,
            "abs_coefficient": np.abs(stage2_coef),
            "direction": np.where(stage2_coef >= 0, "pushes high-SS prediction up", "pushes high-SS prediction down"),
        })
        .sort_values("abs_coefficient", ascending=False)
        .reset_index(drop=True)
    )

    print(f"Stage 2 equation: log1p(SS) = {stage2_intercept:.4f} + sum(coef_j * z_j)")
    print("z_j = median-imputed and standardized feature value.")
    print(f"Coefficient model: {BEST_SS_REGRESSOR_NAME}; trained on high-SS rows only: n={len(high_train)}")
    display(stage2_coef_table.round(5))

    fig, ax = plt.subplots(figsize=(7, 5))
    plot_df = stage2_coef_table.sort_values("coefficient_log1p_standardized")
    colors = np.where(plot_df["coefficient_log1p_standardized"] >= 0, "#4C78A8", "#E45756")
    ax.barh(plot_df["feature"], plot_df["coefficient_log1p_standardized"], color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Ridge coefficient on log1p(SS), standardized features")
    ax.set_title("Stage 2 high-SS Ridge coefficients")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

# Optional: classifier tree feature importance, not coefficients.
classifier_final_for_importance = best_classifier.steps[-1][1]
if hasattr(classifier_final_for_importance, "feature_importances_"):
    classifier_tree_importance = (
        pd.DataFrame({
            "feature": FEATURE_COLS,
            "tree_feature_importance_not_coefficient": classifier_final_for_importance.feature_importances_,
        })
        .sort_values("tree_feature_importance_not_coefficient", ascending=False)
        .reset_index(drop=True)
    )
    print("Stage 1 XGBCls has no linear coefficients. Below is tree feature importance, not coefficient.")
    display(classifier_tree_importance.round(5))


In [ ]:
# Stage 1 SHAP: classifier explanation for SS > 10 mg/L.
classifier_preprocess = Pipeline(best_classifier.steps[:-1])
classifier_final_model = best_classifier.steps[-1][1]

X_train_classifier = pd.DataFrame(
    classifier_preprocess.transform(train_stage[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=train_stage.index,
)
X_test_classifier = pd.DataFrame(
    classifier_preprocess.transform(test_stage[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=test_stage.index,
)

# Use the SHAP explainer that matches the selected classifier.
if hasattr(classifier_final_model, "feature_importances_"):
    classifier_explainer = shap.TreeExplainer(classifier_final_model)
    classifier_shap_explanation_raw = classifier_explainer(X_test_classifier)
    classifier_values = np.asarray(classifier_shap_explanation_raw.values)
    classifier_base_values = np.asarray(classifier_shap_explanation_raw.base_values)

    # For binary tree outputs that return a class dimension, keep class 1 = high SS.
    if classifier_values.ndim == 3:
        classifier_values = classifier_values[:, :, 1]
        if classifier_base_values.ndim == 2:
            classifier_base_values = classifier_base_values[:, 1]
        else:
            classifier_base_values = np.asarray(classifier_base_values).ravel()[1]
        classifier_shap_explanation = shap.Explanation(
            values=classifier_values,
            base_values=classifier_base_values,
            data=X_test_classifier.to_numpy(),
            feature_names=FEATURE_COLS,
        )
    else:
        classifier_shap_explanation = classifier_shap_explanation_raw
elif hasattr(classifier_final_model, "coef_"):
    classifier_explainer = shap.LinearExplainer(classifier_final_model, X_train_classifier)
    classifier_shap_explanation = classifier_explainer(X_test_classifier)
    classifier_values = np.asarray(classifier_shap_explanation.values)
else:
    # Fallback for classifiers such as SVC. Use decision_function when available because
    # it is smoother than hard class labels; otherwise use predict().
    classifier_predict_fn = (
        classifier_final_model.decision_function
        if hasattr(classifier_final_model, "decision_function")
        else classifier_final_model.predict
    )
    classifier_explainer = shap.Explainer(classifier_predict_fn, shap.maskers.Independent(X_train_classifier))
    classifier_shap_explanation = classifier_explainer(X_test_classifier)
    classifier_values = np.asarray(classifier_shap_explanation.values)

if classifier_values.ndim == 3 and classifier_values.shape[-1] == 1:
    classifier_values = classifier_values[:, :, 0]

classifier_shap_importance = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "mean_abs_shap_high_class": np.abs(classifier_values).mean(axis=0),
        "mean_shap_high_class": classifier_values.mean(axis=0),
    })
    .sort_values("mean_abs_shap_high_class", ascending=False)
    .reset_index(drop=True)
)

print("Stage 1 classifier SHAP: positive SHAP pushes the model toward SS > 10 mg/L.")
print("SHAP values explain the selected classifier score for the high-SS decision.")
display(classifier_shap_importance.round(5))

fig, ax = plt.subplots(figsize=(7, 5))
plot_df = classifier_shap_importance.head(20).sort_values("mean_abs_shap_high_class")
ax.barh(plot_df["feature"], plot_df["mean_abs_shap_high_class"], color="#4C78A8")
ax.set_xlabel("Mean |SHAP value| for high-SS class")
ax.set_title(f"Stage 1 SHAP importance | {BEST_SS_CLASSIFIER_NAME} | SS > {LOW_THRESHOLD:g}")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

shap.summary_plot(
    classifier_values,
    X_test_classifier,
    feature_names=FEATURE_COLS,
    max_display=min(20, len(FEATURE_COLS)),
    show=False,
)
plt.title(f"Stage 1 SHAP summary | {BEST_SS_CLASSIFIER_NAME} | high-SS classifier")
plt.tight_layout()
plt.show()


In [ ]:
# Stage 2 SHAP: high-SS regressor explanation.
# Explain only test samples where the two-stage model actually used the high-SS regressor.
regressor_preprocess = Pipeline(best_regressor.steps[:-1])
regressor_final_model = best_regressor.steps[-1][1]

X_train_regressor = pd.DataFrame(
    regressor_preprocess.transform(high_train[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=high_train.index,
)

regressor_explain_mask = best_high_pred == 1
X_regressor_explain_raw = test_stage.loc[regressor_explain_mask, FEATURE_COLS].copy()
regressor_explain_df = test_stage.loc[regressor_explain_mask].copy()

if X_regressor_explain_raw.empty:
    raise ValueError("No test samples were classified as high SS; cannot explain the high-SS regressor on test.")

X_regressor_explain = pd.DataFrame(
    regressor_preprocess.transform(X_regressor_explain_raw),
    columns=FEATURE_COLS,
    index=X_regressor_explain_raw.index,
)

is_cubist_regressor = regressor_final_model.__class__.__module__.lower().startswith("cubist")

if hasattr(regressor_final_model, "coef_"):
    regressor_shap_explainer = shap.LinearExplainer(regressor_final_model, X_train_regressor)
    regressor_shap_explanation = regressor_shap_explainer(X_regressor_explain)
    regressor_values = np.asarray(regressor_shap_explanation.values)
    regressor_coef = np.asarray(regressor_final_model.coef_, dtype=float).ravel()
elif hasattr(regressor_final_model, "feature_importances_") and not is_cubist_regressor:
    regressor_shap_explainer = shap.TreeExplainer(regressor_final_model)
    regressor_shap_explanation = regressor_shap_explainer(X_regressor_explain)
    regressor_values = np.asarray(regressor_shap_explanation.values)
    regressor_coef = np.repeat(np.nan, len(FEATURE_COLS))
else:
    # Cubist is not supported by TreeExplainer, so use model-agnostic SHAP.
    background = X_train_regressor.to_numpy() if is_cubist_regressor else X_train_regressor
    explain_data = X_regressor_explain.to_numpy() if is_cubist_regressor else X_regressor_explain
    predict_fn = (lambda X: regressor_final_model.predict(np.asarray(X))) if is_cubist_regressor else regressor_final_model.predict
    regressor_shap_explainer = shap.Explainer(predict_fn, shap.maskers.Independent(background))
    regressor_shap_explanation = regressor_shap_explainer(explain_data)
    regressor_values = np.asarray(regressor_shap_explanation.values)
    regressor_coef = np.repeat(np.nan, len(FEATURE_COLS))

regressor_shap_importance = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "coefficient_log1p_standardized": regressor_coef,
        "mean_abs_shap_log1p": np.abs(regressor_values).mean(axis=0),
        "mean_shap_log1p": regressor_values.mean(axis=0),
    })
    .sort_values("mean_abs_shap_log1p", ascending=False)
    .reset_index(drop=True)
)

print("Stage 2 regressor SHAP: contribution to log1p(SS) for samples classified as high SS.")
print(f"Explained high-SS test samples: {len(X_regressor_explain)} of {len(test_stage)}")
display(regressor_shap_importance.round(5))

fig, ax = plt.subplots(figsize=(7, 5))
plot_df = regressor_shap_importance.head(20).sort_values("mean_abs_shap_log1p")
ax.barh(plot_df["feature"], plot_df["mean_abs_shap_log1p"], color="#4C78A8")
ax.set_xlabel("Mean |SHAP value| on log1p(SS)")
ax.set_title(f"Stage 2 SHAP importance | {BEST_SS_REGRESSOR_NAME} | high-SS regressor")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

shap.summary_plot(
    regressor_values,
    X_regressor_explain,
    feature_names=FEATURE_COLS,
    max_display=min(20, len(FEATURE_COLS)),
    show=False,
)
plt.title(f"Stage 2 SHAP summary | {BEST_SS_REGRESSOR_NAME} | high-SS regressor")
plt.tight_layout()
plt.show()


In [ ]:
# Local SHAP explanations for the highest observed SS sample in the test set.
highest_test_idx = test_stage[TARGET_COL].astype(float).idxmax()
highest_pos_all = list(test_stage.index).index(highest_test_idx)

local_meta = test_stage.loc[[highest_test_idx], [
    c for c in ["sample_id", "station_canonical", "year_be", "field_date", "sat_date", "abs_date_diff_days", TARGET_COL, "is_high"]
    if c in test_stage.columns
]].copy()
local_meta["predicted_is_high"] = int(best_high_pred[highest_pos_all])
local_meta["predicted_ss"] = float(best_combined_pred[highest_pos_all])
local_meta["residual"] = local_meta["predicted_ss"] - local_meta[TARGET_COL]

if hasattr(best_classifier, "predict_proba"):
    local_meta["predicted_high_probability"] = float(best_classifier.predict_proba(test_stage.loc[[highest_test_idx], FEATURE_COLS])[:, 1][0])

print("Local SHAP example: highest observed SS sample in test set")
display(local_meta.round(4))

print("Stage 1 classifier local explanation: why classified as high/low SS")
shap.plots.waterfall(classifier_shap_explanation[highest_pos_all], max_display=min(12, len(FEATURE_COLS)), show=False)
plt.title("Stage 1 local SHAP: high-SS classifier")
plt.tight_layout()
plt.show()

if int(best_high_pred[highest_pos_all]) == 1 and highest_test_idx in X_regressor_explain.index:
    highest_pos_regressor = list(X_regressor_explain.index).index(highest_test_idx)
    print("Stage 2 regressor local explanation: why this high-SS value was predicted")
    shap.plots.waterfall(regressor_shap_explanation[highest_pos_regressor], max_display=min(12, len(FEATURE_COLS)), show=False)
    plt.title("Stage 2 local SHAP: high-SS regressor on log1p(SS)")
    plt.tight_layout()
    plt.show()
else:
    print("The highest-observed SS sample was classified as low, so the high-SS regressor was not used for this sample.")


## Feature-Set SHAP Comparison: S2 Only vs S2+Context

This section repeats the SS two-stage workflow for two feature sets:

- `S2 only`: Sentinel-2 bands and indices only.
- `S2 + Context`: Sentinel-2 features plus selected rainfall, runoff, wind, season, and long-term water-context features.

For each feature set, the notebook selects the best actual two-stage model by original-scale R2 on the temporal test set. SHAP is then calculated separately for:

1. Stage 1 classifier: explains whether the model classifies a sample as `SS > 10 mg/L`.
2. Stage 2 high-SS regressor: explains the predicted `log1p(SS)` for samples classified as high SS.


In [ ]:
# Define S2-only and S2+Context feature sets for SS SHAP comparison.
CONTEXT_FEATURE_COLS = [
    "rain_gsmap_prior_24h_mm", "rain_gsmap_prior_72h_mm", "rain_gsmap_prior_168h_mm",
    "era5h_runoff_prior_24h_mm", "era5h_runoff_prior_72h_mm", "era5h_runoff_prior_168h_mm",
    "era5h_wind_mean_prior_24h_m_s", "era5h_wind_max_prior_24h_m_s",
    "doy_sin", "doy_cos",
    "wc_water_frac_500m", "jrc_occurrence_mean_500m",
]

missing_context_cols = [c for c in CONTEXT_FEATURE_COLS if c not in df.columns]
if missing_context_cols:
    raise KeyError(f"Missing context feature columns: {missing_context_cols}")

SS_SHAP_FEATURE_SETS = {
    "S2 only": FEATURE_COLS,
    "S2 + Context": FEATURE_COLS + CONTEXT_FEATURE_COLS,
}

def ss_feature_group(feature_name):
    return "S2" if feature_name in FEATURE_COLS else "Context"

feature_set_summary = pd.DataFrame({
    "feature_set": list(SS_SHAP_FEATURE_SETS.keys()),
    "n_features": [len(cols) for cols in SS_SHAP_FEATURE_SETS.values()],
    "n_s2_features": [sum(c in FEATURE_COLS for c in cols) for cols in SS_SHAP_FEATURE_SETS.values()],
    "n_context_features": [sum(c in CONTEXT_FEATURE_COLS for c in cols) for cols in SS_SHAP_FEATURE_SETS.values()],
})
display(feature_set_summary)


In [ ]:
# Evaluate and select the best actual two-stage model for each feature set.
def evaluate_two_stage_feature_set(feature_set_name, feature_cols):
    X_train_class = train_stage[feature_cols]
    y_train_class = train_stage["is_high"]
    X_test_class = test_stage[feature_cols]
    y_test_original = test_stage[TARGET_COL].values

    fitted_classifiers = {}
    classifier_preds = {}
    classifier_proba = {}
    for classifier_name, classifier in make_classifiers(RANDOM_STATE).items():
        classifier.fit(X_train_class, y_train_class)
        pred = np.asarray(classifier.predict(X_test_class)).ravel().astype(int)
        fitted_classifiers[classifier_name] = classifier
        classifier_preds[classifier_name] = pred
        if hasattr(classifier, "predict_proba"):
            try:
                classifier_proba[classifier_name] = classifier.predict_proba(X_test_class)[:, 1]
            except Exception:
                classifier_proba[classifier_name] = np.repeat(np.nan, len(test_stage))

    fitted_regressors = {}
    regressor_preds = {}
    for regressor_name, regressor in make_high_regressors(RANDOM_STATE).items():
        regressor.fit(high_train[feature_cols], high_train[TARGET_LOG_COL])
        pred_log = np.asarray(regressor.predict(X_test_class)).ravel()
        fitted_regressors[regressor_name] = regressor
        regressor_preds[regressor_name] = np.expm1(np.clip(pred_log, 0, None))

    rows = []
    for classifier_name, high_pred in classifier_preds.items():
        for regressor_name, reg_pred in regressor_preds.items():
            combined_pred = np.where(high_pred == 1, reg_pred, low_value)
            metric = two_stage_metrics_with_log(y_test_original, combined_pred)
            rows.append({
                "feature_set": feature_set_name,
                "classifier": classifier_name,
                "regressor": regressor_name,
                "n_features": len(feature_cols),
                "n_train_classifier": len(train_stage),
                "n_train_high_regressor": len(high_train),
                "n_test": len(test_stage),
                "pred_high": int(high_pred.sum()),
                "true_high": int(test_stage["is_high"].sum()),
                **metric,
            })

    results_df = pd.DataFrame(rows).sort_values(["original_r2", "spearman_r"], ascending=False).reset_index(drop=True)
    best_row = results_df.iloc[0].copy()
    best_classifier = fitted_classifiers[best_row["classifier"]]
    best_regressor = fitted_regressors[best_row["regressor"]]
    best_high_pred = classifier_preds[best_row["classifier"]]
    best_reg_pred = regressor_preds[best_row["regressor"]]
    best_combined_pred = np.where(best_high_pred == 1, best_reg_pred, low_value)

    return {
        "feature_set": feature_set_name,
        "feature_cols": feature_cols,
        "results": results_df,
        "best_row": best_row,
        "best_classifier": best_classifier,
        "best_regressor": best_regressor,
        "best_high_pred": best_high_pred,
        "best_combined_pred": best_combined_pred,
        "classifier_proba": classifier_proba.get(best_row["classifier"], None),
    }

feature_set_runs = {}
for feature_set_name, feature_cols in SS_SHAP_FEATURE_SETS.items():
    feature_set_runs[feature_set_name] = evaluate_two_stage_feature_set(feature_set_name, feature_cols)

feature_set_best_summary = pd.DataFrame([
    run["best_row"].to_dict() for run in feature_set_runs.values()
])

display(feature_set_best_summary[[
    "feature_set", "classifier", "regressor", "n_features", "n_train_classifier", "n_train_high_regressor",
    "n_test", "pred_high", "true_high", "original_r2", "original_rmse", "log_r2", "log_rmse", "spearman_r", "bias",
]].round(4))


In [ ]:
# Helper functions for SHAP on sklearn pipelines.
def pipeline_prepared_frame(pipeline, raw_X, feature_cols):
    if isinstance(pipeline, Pipeline) and len(pipeline.steps) > 1:
        preprocess = Pipeline(pipeline.steps[:-1])
        prepared = preprocess.transform(raw_X)
    else:
        prepared = raw_X.to_numpy()
    return pd.DataFrame(prepared, columns=feature_cols, index=raw_X.index)

def final_estimator(pipeline):
    return pipeline.steps[-1][1] if isinstance(pipeline, Pipeline) else pipeline

def explain_classifier_high_class(classifier_pipeline, X_train_raw, X_test_raw, feature_cols):
    X_train_prepared = pipeline_prepared_frame(classifier_pipeline, X_train_raw, feature_cols)
    X_test_prepared = pipeline_prepared_frame(classifier_pipeline, X_test_raw, feature_cols)
    estimator = final_estimator(classifier_pipeline)

    if hasattr(estimator, "feature_importances_"):
        explanation_raw = shap.TreeExplainer(estimator)(X_test_prepared)
        values = np.asarray(explanation_raw.values)
        base_values = np.asarray(explanation_raw.base_values)
        if values.ndim == 3:
            values = values[:, :, 1]
            if base_values.ndim == 2:
                base_values = base_values[:, 1]
            elif base_values.size > 1:
                base_values = np.repeat(base_values.ravel()[1], len(X_test_prepared))
        else:
            base_values = base_values.ravel() if base_values.ndim else np.repeat(float(base_values), len(X_test_prepared))
        explanation = shap.Explanation(
            values=values,
            base_values=base_values,
            data=X_test_prepared.to_numpy(),
            feature_names=feature_cols,
        )
    elif hasattr(estimator, "coef_"):
        explanation = shap.LinearExplainer(estimator, X_train_prepared)(X_test_prepared)
        values = np.asarray(explanation.values)
    else:
        masker = shap.maskers.Independent(X_train_prepared)
        explanation = shap.Explainer(estimator.predict, masker)(X_test_prepared)
        values = np.asarray(explanation.values)

    return explanation, values, X_test_prepared

def explain_regressor_model(regressor_pipeline, X_train_raw, X_explain_raw, feature_cols):
    X_train_prepared = pipeline_prepared_frame(regressor_pipeline, X_train_raw, feature_cols)
    X_explain_prepared = pipeline_prepared_frame(regressor_pipeline, X_explain_raw, feature_cols)
    estimator = final_estimator(regressor_pipeline)
    is_cubist = estimator.__class__.__module__.lower().startswith("cubist")

    if hasattr(estimator, "coef_"):
        explanation = shap.LinearExplainer(estimator, X_train_prepared)(X_explain_prepared)
    elif hasattr(estimator, "feature_importances_") and not is_cubist:
        explanation = shap.TreeExplainer(estimator)(X_explain_prepared)
    else:
        if is_cubist:
            background = X_train_prepared.to_numpy()
            explain_data = X_explain_prepared.to_numpy()
            predict_fn = lambda X: estimator.predict(np.asarray(X))
        else:
            background = X_train_prepared
            explain_data = X_explain_prepared
            predict_fn = estimator.predict
        masker = shap.maskers.Independent(background)
        explanation = shap.Explainer(predict_fn, masker)(explain_data)

    values = np.asarray(explanation.values)
    if values.ndim == 3 and values.shape[-1] == 1:
        values = values[:, :, 0]
    return explanation, values, X_explain_prepared


In [ ]:
# Calculate SHAP for each feature set and stage.
feature_set_shap_outputs = {}
classifier_shap_rows = []
regressor_shap_rows = []
group_shap_rows = []

for feature_set_name, run in feature_set_runs.items():
    feature_cols = run["feature_cols"]
    best_classifier = run["best_classifier"]
    best_regressor = run["best_regressor"]
    best_high_pred = run["best_high_pred"]

    X_train_class_raw = train_stage[feature_cols]
    X_test_class_raw = test_stage[feature_cols]

    cls_explanation, cls_values, X_test_cls_prepared = explain_classifier_high_class(
        best_classifier, X_train_class_raw, X_test_class_raw, feature_cols
    )

    cls_importance = pd.DataFrame({
        "feature_set": feature_set_name,
        "stage": "Stage 1 classifier",
        "model": run["best_row"]["classifier"],
        "feature": feature_cols,
        "feature_group": [ss_feature_group(c) for c in feature_cols],
        "mean_abs_shap": np.abs(cls_values).mean(axis=0),
        "mean_shap": cls_values.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    classifier_shap_rows.append(cls_importance)

    cls_group = (
        cls_importance.groupby(["feature_set", "stage", "feature_group"], as_index=False)
        .agg(total_mean_abs_shap=("mean_abs_shap", "sum"), mean_abs_shap=("mean_abs_shap", "mean"), n_features=("feature", "count"))
    )
    group_shap_rows.append(cls_group)

    regressor_explain_mask = best_high_pred == 1
    X_reg_explain_raw = test_stage.loc[regressor_explain_mask, feature_cols]
    if X_reg_explain_raw.empty:
        continue

    reg_explanation, reg_values, X_reg_prepared = explain_regressor_model(
        best_regressor,
        high_train[feature_cols],
        X_reg_explain_raw,
        feature_cols,
    )

    reg_estimator = final_estimator(best_regressor)
    reg_coef = np.asarray(getattr(reg_estimator, "coef_", np.repeat(np.nan, len(feature_cols))), dtype=float).ravel()
    if len(reg_coef) != len(feature_cols):
        reg_coef = np.repeat(np.nan, len(feature_cols))

    reg_importance = pd.DataFrame({
        "feature_set": feature_set_name,
        "stage": "Stage 2 high-SS regressor",
        "model": run["best_row"]["regressor"],
        "feature": feature_cols,
        "feature_group": [ss_feature_group(c) for c in feature_cols],
        "coefficient_log1p_standardized": reg_coef,
        "mean_abs_shap": np.abs(reg_values).mean(axis=0),
        "mean_shap": reg_values.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    regressor_shap_rows.append(reg_importance)

    reg_group = (
        reg_importance.groupby(["feature_set", "stage", "feature_group"], as_index=False)
        .agg(total_mean_abs_shap=("mean_abs_shap", "sum"), mean_abs_shap=("mean_abs_shap", "mean"), n_features=("feature", "count"))
    )
    group_shap_rows.append(reg_group)

    feature_set_shap_outputs[feature_set_name] = {
        "classifier_explanation": cls_explanation,
        "classifier_values": cls_values,
        "classifier_X": X_test_cls_prepared,
        "classifier_importance": cls_importance,
        "regressor_explanation": reg_explanation,
        "regressor_values": reg_values,
        "regressor_X": X_reg_prepared,
        "regressor_importance": reg_importance,
        "regressor_explain_index": X_reg_explain_raw.index,
    }

classifier_shap_comparison = pd.concat(classifier_shap_rows, ignore_index=True)
regressor_shap_comparison = pd.concat(regressor_shap_rows, ignore_index=True)
group_shap_comparison = pd.concat(group_shap_rows, ignore_index=True)

print("Top Stage 1 classifier SHAP features by feature set")
display(classifier_shap_comparison.groupby("feature_set", group_keys=False).head(12).round(5))

print("Top Stage 2 high-SS regressor SHAP features by feature set")
display(regressor_shap_comparison.groupby("feature_set", group_keys=False).head(12).round(5))

print("Group-level SHAP comparison: S2 vs Context")
display(group_shap_comparison.sort_values(["feature_set", "stage", "total_mean_abs_shap"], ascending=[True, True, False]).round(5))


In [ ]:
# Plot SHAP comparison for S2 only and S2+Context.
for feature_set_name, shap_out in feature_set_shap_outputs.items():
    for stage_key, title_label, value_col in [
        ("classifier", "Stage 1 classifier", "mean_abs_shap"),
        ("regressor", "Stage 2 high-SS regressor", "mean_abs_shap"),
    ]:
        importance = shap_out[f"{stage_key}_importance"]
        values = shap_out[f"{stage_key}_values"]
        X_prepared = shap_out[f"{stage_key}_X"]

        print(f"{feature_set_name} | {title_label}")
        display(importance.head(12).round(5))

        fig, ax = plt.subplots(figsize=(7, 5))
        plot_df = importance.head(15).sort_values(value_col)
        colors = np.where(plot_df["feature_group"].eq("S2"), "#4C78A8", "#F58518")
        ax.barh(plot_df["feature"], plot_df[value_col], color=colors)
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(f"{feature_set_name} | {title_label} SHAP importance")
        ax.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        plt.show()

        shap.summary_plot(
            values,
            X_prepared,
            feature_names=list(X_prepared.columns),
            max_display=min(15, X_prepared.shape[1]),
            show=False,
        )
        plt.title(f"{feature_set_name} | {title_label} SHAP summary")
        plt.tight_layout()
        plt.show()


## Feature-Set Coefficient Comparison: S2 Only vs S2+Context

This section compares coefficients for the best Stage 2 high-SS regressors selected in the feature-set SHAP comparison.

Important: the Stage 1 classifiers selected here are tree models (`XGBCls` for S2 only and `RandomForestCls` for S2+Context), so they do not have linear coefficients. For Stage 1, use SHAP or tree feature importance. Coefficients below apply only to Stage 2 high-SS regression, which estimates `log1p(SS)` for samples classified as `SS > 10 mg/L`.


In [ ]:
# Compare Stage 2 coefficients for S2 only vs S2+Context best two-stage models.
if "feature_set_runs" not in globals():
    raise RuntimeError("Run the Feature-Set SHAP Comparison section before this coefficient comparison.")

feature_set_coef_tables = []
feature_set_classifier_importance_tables = []

for feature_set_name, run in feature_set_runs.items():
    feature_cols = run["feature_cols"]
    best_row = run["best_row"]
    best_classifier = run["best_classifier"]
    best_regressor = run["best_regressor"]

    reg_estimator = final_estimator(best_regressor) if "final_estimator" in globals() else best_regressor.steps[-1][1]
    if hasattr(reg_estimator, "coef_"):
        coef = np.asarray(reg_estimator.coef_, dtype=float).ravel()
        if len(coef) != len(feature_cols):
            coef = coef.reshape(-1)[:len(feature_cols)]
        intercept = getattr(reg_estimator, "intercept_", np.nan)
        try:
            intercept_value = float(np.asarray(intercept).ravel()[0])
        except Exception:
            intercept_value = np.nan

        coef_table = pd.DataFrame({
            "feature_set": feature_set_name,
            "stage": "Stage 2 high-SS regressor",
            "model": best_row["regressor"],
            "feature": feature_cols,
            "feature_group": [ss_feature_group(c) if "ss_feature_group" in globals() else ("S2" if c in FEATURE_COLS else "Context") for c in feature_cols],
            "coefficient_log1p_standardized": coef,
            "abs_coefficient": np.abs(coef),
            "direction": np.where(coef >= 0, "pushes high-SS prediction up", "pushes high-SS prediction down"),
            "intercept_log1p": intercept_value,
            "n_train_high_regressor": len(high_train),
            "best_original_r2": best_row["original_r2"],
            "best_original_rmse": best_row["original_rmse"],
        }).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
        feature_set_coef_tables.append(coef_table)
    else:
        print(f"{feature_set_name}: {best_row['regressor']} has no linear coefficient.")

    cls_estimator = final_estimator(best_classifier) if "final_estimator" in globals() else best_classifier.steps[-1][1]
    if hasattr(cls_estimator, "feature_importances_"):
        cls_importance = pd.DataFrame({
            "feature_set": feature_set_name,
            "stage": "Stage 1 classifier",
            "model": best_row["classifier"],
            "feature": feature_cols,
            "feature_group": [ss_feature_group(c) if "ss_feature_group" in globals() else ("S2" if c in FEATURE_COLS else "Context") for c in feature_cols],
            "tree_feature_importance_not_coefficient": cls_estimator.feature_importances_,
        }).sort_values("tree_feature_importance_not_coefficient", ascending=False).reset_index(drop=True)
        feature_set_classifier_importance_tables.append(cls_importance)

feature_set_stage2_coefficients = pd.concat(feature_set_coef_tables, ignore_index=True)
feature_set_stage1_tree_importance = pd.concat(feature_set_classifier_importance_tables, ignore_index=True)

print("Best Stage 2 coefficient table by feature set")
display(feature_set_stage2_coefficients.groupby("feature_set", group_keys=False).head(15).round(5))

stage2_coef_group_summary = (
    feature_set_stage2_coefficients
    .groupby(["feature_set", "feature_group"], as_index=False)
    .agg(
        total_abs_coefficient=("abs_coefficient", "sum"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        n_features=("feature", "count"),
    )
    .sort_values(["feature_set", "total_abs_coefficient"], ascending=[True, False])
)
print("Stage 2 coefficient group summary: S2 vs Context")
display(stage2_coef_group_summary.round(5))

print("Stage 1 classifiers are tree models and do not have coefficients. Below is tree feature importance, not coefficient.")
display(feature_set_stage1_tree_importance.groupby("feature_set", group_keys=False).head(12).round(5))


In [ ]:
# Plot Stage 2 coefficients by feature set.
for feature_set_name in feature_set_stage2_coefficients["feature_set"].drop_duplicates():
    coef_table = feature_set_stage2_coefficients.loc[
        feature_set_stage2_coefficients["feature_set"].eq(feature_set_name)
    ].copy()
    model_name = coef_table["model"].iloc[0]
    metric_r2 = coef_table["best_original_r2"].iloc[0]
    metric_rmse = coef_table["best_original_rmse"].iloc[0]

    print(f"{feature_set_name} | Stage 2 coefficient model: {model_name} | R2={metric_r2:.4f} | RMSE={metric_rmse:.4f}")
    display(coef_table.head(15).round(5))

    fig, ax = plt.subplots(figsize=(7, 6 if len(coef_table) <= 16 else 8))
    plot_df = coef_table.head(20).sort_values("coefficient_log1p_standardized")
    colors = np.where(plot_df["feature_group"].eq("S2"), "#4C78A8", "#F58518")
    ax.barh(plot_df["feature"], plot_df["coefficient_log1p_standardized"], color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Coefficient on log1p(SS), standardized features")
    ax.set_title(f"{feature_set_name} | Stage 2 {model_name} coefficients")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


## Two-stage Threshold Feature Search Results

This section reports a broad search for the best `SS_mg_l_clean` two-stage model under the current temporal split:

- Train years: 2562-2566
- Test years: 2567-2568
- Sentinel-2 match window: `abs_date_diff_days <= 1`
- Model structure: Stage 1 low/high classifier + Stage 2 high-SS regressor on `log1p(SS)`
- Search size already executed: 168,960 configurations
- Main ranking metric: original-scale `R2`, with RMSE/log metrics shown for checking

Important interpretation:
- `cut_scope = train_only` keeps all test samples. This is the cleaner evaluation if you want the 2567-2568 test set untouched.
- `cut_scope = train_test` applies the SS upper threshold to both train and test. This is a limited-domain diagnostic and must be reported as such.


In [ ]:
# Load the completed SS two-stage threshold feature-search results.
# This is a report cell so you do not accidentally rerun the full 168,960-config grid.

from pathlib import Path
import pandas as pd

SS_SEARCH_RESULT_PATH = Path(r".\Resources\notebooks\Tur_SS\SS\notebook_outputs\ss_two_stage_threshold_feature_search_train2562_2566_test2567_2568.csv")
SS_SEARCH_PRED_PATH = Path(r".\Resources\notebooks\Tur_SS\SS\notebook_outputs\ss_two_stage_threshold_feature_search_best_predictions.csv")

if not SS_SEARCH_RESULT_PATH.exists():
    raise FileNotFoundError(f"Missing result file: {SS_SEARCH_RESULT_PATH}")

ss_search_results = pd.read_csv(SS_SEARCH_RESULT_PATH)
ss_search_results = ss_search_results.sort_values(
    ["original_r2", "log_r2", "original_rmse"],
    ascending=[False, False, True],
).reset_index(drop=True)

ss_search_show_cols = [
    "feature_set", "model", "max_ss_mg_l", "cut_scope", "low_threshold", "prob_threshold",
    "n_features", "n_train", "n_train_high", "n_test", "n_test_high", "n_test_removed",
    "original_r2", "original_rmse", "original_mae", "log_r2", "log_rmse", "spearman_r", "bias",
    "classifier_balanced_accuracy", "pred_high",
]

print("SS two-stage threshold feature search result file")
print(SS_SEARCH_RESULT_PATH)
print("Total configurations:", len(ss_search_results))

print("Best overall models by original-scale R2")
display(ss_search_results[ss_search_show_cols].head(30).round(4))

print("Best models with untouched test set (cut_scope = train_only)")
display(ss_search_results.query("cut_scope == 'train_only'")[ss_search_show_cols].head(20).round(4))

print("Best limited-domain models (cut_scope = train_test)")
display(ss_search_results.query("cut_scope == 'train_test'")[ss_search_show_cols].head(20).round(4))

print("Best model inside each feature set")
ss_best_by_feature_set = (
    ss_search_results
    .sort_values(["feature_set", "original_r2", "log_r2", "original_rmse"], ascending=[True, False, False, True])
    .groupby("feature_set", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
display(ss_best_by_feature_set[ss_search_show_cols].round(4))

print("Best prediction file")
print(SS_SEARCH_PRED_PATH)
if SS_SEARCH_PRED_PATH.exists():
    ss_best_predictions = pd.read_csv(SS_SEARCH_PRED_PATH)
    display(ss_best_predictions.sort_values("observed_ss", ascending=False).head(20).round(4))


## Stage-1 Classifier Hyperparameter Tuning Results

This section reports a focused tuning experiment for the two-stage SS model.  
Only the Stage-1 low/high classifier and its probability threshold were tuned. The rest of the setup was locked to isolate the classifier effect:

- Data: relaxed v4, within 1 day match only
- Split: train 2562-2566, test 2567-2568
- Train-domain cut: remove train samples with SS > 130 mg/L; test remains untouched
- Low/high split: SS > 12 mg/L is treated as high SS
- Feature set: S2_23_plus_water_QA
- Stage-2 regressor: PLS_10 on log1p(SS) for high-SS samples

Two rankings are shown:

- `validation_selected`: probability threshold selected by validation year 2566, then evaluated on 2567-2568. This is the cleaner leakage-aware setting.
- `test_threshold_diagnostic`: probability threshold selected on the test set. This is only a diagnostic upper-bound and should not be reported as the final leakage-controlled result.


In [ ]:
# Report Stage-1 classifier hyperparameter tuning results.
# The full tuning was already run and saved to CSV so this cell is fast and reproducible for reporting.

from pathlib import Path
import pandas as pd

SS_STAGE1_TUNING_PATH = Path(r".\Resources\notebooks\Tur_SS\SS\notebook_outputs\ss_stage1_classifier_hyperparameter_tuning_train2562_2566_test2567_2568.csv")
SS_STAGE1_TUNING_PRED_PATH = Path(r".\Resources\notebooks\Tur_SS\SS\notebook_outputs\ss_stage1_classifier_hyperparameter_tuning_best_predictions.csv")

if not SS_STAGE1_TUNING_PATH.exists():
    raise FileNotFoundError(f"Missing result file: {SS_STAGE1_TUNING_PATH}")

stage1_tuning = pd.read_csv(SS_STAGE1_TUNING_PATH)

show_cols = [
    "selection_mode", "classifier", "classifier_config", "prob_threshold",
    "n_final_train", "n_test", "classifier_balanced_accuracy", "classifier_f1_high",
    "test_original_r2", "test_original_rmse", "test_original_mae",
    "test_log_r2", "test_log_rmse", "test_spearman_r", "test_bias",
    "val_original_r2", "val_log_r2",
]

print("Stage-1 classifier tuning result file")
print(SS_STAGE1_TUNING_PATH)
print("Rows:", len(stage1_tuning))
print("Classifier specs:", stage1_tuning[["classifier", "classifier_config"]].drop_duplicates().shape[0])
print("Classifiers tried:", sorted(stage1_tuning["classifier"].unique().tolist()))

print("Best validation-selected configs ranked by final test R2 for diagnosis")
display(
    stage1_tuning
    .query("selection_mode == 'validation_selected'")
    .sort_values(["test_original_r2", "test_log_r2", "test_original_rmse"], ascending=[False, False, True])
    [show_cols]
    .head(20)
    .round(4)
)

print("Config selected strictly by validation-year R2, then evaluated on final test")
display(
    stage1_tuning
    .query("selection_mode == 'validation_selected'")
    .sort_values(["val_original_r2", "val_log_r2", "val_original_rmse"], ascending=[False, False, True])
    [show_cols]
    .head(10)
    .round(4)
)

print("Diagnostic upper-bound: best configs when probability threshold is chosen on the test set")
display(
    stage1_tuning
    .query("selection_mode == 'test_threshold_diagnostic'")
    .sort_values(["test_original_r2", "test_log_r2", "test_original_rmse"], ascending=[False, False, True])
    [show_cols]
    .head(20)
    .round(4)
)

print("Best prediction file")
print(SS_STAGE1_TUNING_PRED_PATH)
if SS_STAGE1_TUNING_PRED_PATH.exists():
    stage1_best_predictions = pd.read_csv(SS_STAGE1_TUNING_PRED_PATH)
    display(stage1_best_predictions.sort_values(["selection_label", "observed_ss"], ascending=[True, False]).head(30).round(4))


<!-- SS_TWO_STAGE_FIXED_MODEL_COMPARISON_START -->
## Two-Stage Fixed Model Comparison: S2 Only vs S2 + Context

This section compares the requested Stage 2 regression families under one fixed two-stage design.

- Train: 2562-2566; test: 2567-2568
- Sentinel-2 match: within 1 day
- Applicability range: SS <= 130 mg/L
- Low/high cutoff: 15 mg/L
- Stage 1: balanced Logistic Regression
- Stage 2: regression on `log1p(SS)` using high-SS training samples only
- Final prediction: soft probability-weighted gating
- Feature sets: the same 23 S2 features, with or without the same 10 water/QA context features

Hyperparameters are fixed before test evaluation. This section does not select a separate best hyperparameter from the test set for each model.


In [ ]:
# Independent setup for the fixed two-stage model comparison
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, HuberRegressor, LogisticRegression, Ridge
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SSCMP_DATA_PATH = Path(
    r".\Data\tabular_data_14\train\v4\rswq_model_input_relaxed_s2_context_2562_2568.csv"
)
SSCMP_TARGET = "SS_mg_l_clean"
SSCMP_SEED = 19
SSCMP_LOW_THRESHOLD = 15.0
SSCMP_MAX_SS = 130.0
SSCMP_MAX_MATCH_DAYS = 1


def sscmp_ratio(a, b, eps=1e-12):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    out = np.where(np.abs(b) > eps, a / b, np.nan)
    return np.where(np.isfinite(out), out, np.nan)


def sscmp_nd(a, b, eps=1e-12):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    denominator = a + b
    out = np.where(np.abs(denominator) > eps, (a - b) / denominator, np.nan)
    return np.where(np.isfinite(out), out, np.nan)


def sscmp_add_s2_features(data):
    data = data.copy()
    data["NDWI1"] = sscmp_nd(data["B3_median"], data["B8_median"])
    data["NDWI2"] = sscmp_nd(data["B8_median"], data["B11_median"])
    data["NDWI3"] = sscmp_nd(data["B8_median"], data["B12_median"])
    data["B4_B3_ratio"] = sscmp_ratio(data["B4_median"], data["B3_median"])
    data["B8_B4_ratio"] = sscmp_ratio(data["B8_median"], data["B4_median"])
    data["NIR_Blue_ratio"] = sscmp_ratio(data["B8_median"], data["B2_median"])
    data["NIR_Green_ratio"] = sscmp_ratio(data["B8_median"], data["B3_median"])
    data["NIR_Red_ratio"] = sscmp_ratio(data["B8_median"], data["B4_median"])
    data["Red_NIR_ND"] = sscmp_nd(data["B4_median"], data["B8_median"])
    return data


SSCMP_S2_FEATURES = [
    "B1_median", "B2_median", "B3_median", "B4_median",
    "B5_median", "B6_median", "B7_median", "B8_median",
    "B8A_median", "B9_median", "B11_median", "B12_median",
    "NDTI_median", "NDWI_median", "NDWI1", "NDWI3",
    "B4_B3_ratio", "B8_B4_ratio", "NDWI2", "NIR_Blue_ratio",
    "NIR_Green_ratio", "NIR_Red_ratio", "Red_NIR_ND",
]
SSCMP_CONTEXT_FEATURES = [
    "valid_water_pixel_count", "water_pixel_fraction", "candidate_water_count",
    "water_pixel_pct", "local_cloud_pct", "scene_cloud_pct",
    "jrc_occurrence_mean_500m", "wc_water_frac_500m",
    "wc_water_frac_1000m", "wc_water_frac_5000m",
]
SSCMP_FEATURE_SETS = {
    "S2 only": SSCMP_S2_FEATURES,
    "S2 + Context": SSCMP_S2_FEATURES + SSCMP_CONTEXT_FEATURES,
}

sscmp_df = sscmp_add_s2_features(pd.read_csv(SSCMP_DATA_PATH, low_memory=False))
sscmp_year = pd.to_numeric(sscmp_df["year_be"], errors="coerce")
sscmp_days = pd.to_numeric(sscmp_df["abs_date_diff_days"], errors="coerce")
sscmp_target = pd.to_numeric(sscmp_df[SSCMP_TARGET], errors="coerce")
sscmp_df["SS_log1p"] = np.log1p(sscmp_target)

sscmp_required = sorted(set(sum(SSCMP_FEATURE_SETS.values(), [])))
sscmp_missing = [column for column in sscmp_required if column not in sscmp_df.columns]
if sscmp_missing:
    raise KeyError(f"Missing comparison features: {sscmp_missing}")

sscmp_base = (
    sscmp_target.notna()
    & sscmp_days.le(SSCMP_MAX_MATCH_DAYS)
    & sscmp_target.le(SSCMP_MAX_SS)
)
sscmp_train = sscmp_df.loc[sscmp_base & sscmp_year.between(2562, 2566)].copy()
sscmp_test = sscmp_df.loc[sscmp_base & sscmp_year.between(2567, 2568)].copy()
sscmp_train["is_high"] = sscmp_train[SSCMP_TARGET].gt(SSCMP_LOW_THRESHOLD).astype(int)
sscmp_test["is_high"] = sscmp_test[SSCMP_TARGET].gt(SSCMP_LOW_THRESHOLD).astype(int)
sscmp_high_train = sscmp_train.loc[sscmp_train["is_high"].eq(1)].copy()
sscmp_low_value = float(
    sscmp_train.loc[sscmp_train["is_high"].eq(0), SSCMP_TARGET].median()
)

sscmp_split_summary = pd.DataFrame([
    {
        "split": "train_2562_2566",
        "rows": len(sscmp_train),
        "stations": sscmp_train["station_canonical"].nunique(),
        "low_count": int(sscmp_train["is_high"].eq(0).sum()),
        "high_count": int(sscmp_train["is_high"].eq(1).sum()),
        "target_min": sscmp_train[SSCMP_TARGET].min(),
        "target_median": sscmp_train[SSCMP_TARGET].median(),
        "target_max": sscmp_train[SSCMP_TARGET].max(),
    },
    {
        "split": "test_2567_2568",
        "rows": len(sscmp_test),
        "stations": sscmp_test["station_canonical"].nunique(),
        "low_count": int(sscmp_test["is_high"].eq(0).sum()),
        "high_count": int(sscmp_test["is_high"].eq(1).sum()),
        "target_min": sscmp_test[SSCMP_TARGET].min(),
        "target_median": sscmp_test[SSCMP_TARGET].median(),
        "target_max": sscmp_test[SSCMP_TARGET].max(),
    },
])

print("Fixed two-stage comparison setup")
display(sscmp_split_summary.round(4))
print("Low-group training median:", round(sscmp_low_value, 4))
print("Feature counts:", {name: len(cols) for name, cols in SSCMP_FEATURE_SETS.items()})


In [ ]:
# Fixed Stage 1 classifier and requested Stage 2 regression models
def sscmp_classifier():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=SSCMP_SEED,
        )),
    ])


def sscmp_regressors():
    return {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "ElasticNet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                alpha=0.01, l1_ratio=0.15, max_iter=20000,
                random_state=SSCMP_SEED,
            )),
        ]),
        "HuberRegressor": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", HuberRegressor(
                epsilon=1.35, alpha=0.0001, max_iter=2000,
            )),
        ]),
        "PLS_10": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=10)),
        ]),
        "LinearSVR": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearSVR(
                C=1.0, epsilon=0.3, loss="epsilon_insensitive",
                dual="auto", max_iter=100000, random_state=SSCMP_SEED,
            )),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=500, min_samples_leaf=2,
                random_state=SSCMP_SEED, n_jobs=-1,
            )),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=500, min_samples_leaf=2,
                random_state=SSCMP_SEED, n_jobs=-1,
            )),
        ]),
        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=500, max_depth=2, learning_rate=0.03,
                subsample=0.85, colsample_bytree=0.85, reg_lambda=5.0,
                objective="reg:squarederror", random_state=SSCMP_SEED,
                n_jobs=-1,
            )),
        ]),
        "CatBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=700, depth=3, learning_rate=0.03,
                loss_function="RMSE", random_seed=SSCMP_SEED,
                verbose=False, allow_writing_files=False,
            )),
        ]),
    }


SSCMP_MODEL_CONFIGS = {
    "Ridge": "alpha=1.0",
    "ElasticNet": "alpha=0.01; l1_ratio=0.15",
    "HuberRegressor": "epsilon=1.35; alpha=0.0001",
    "PLS_10": "n_components=10",
    "LinearSVR": "C=1.0; epsilon=0.3; linear kernel",
    "RandomForest": "n_estimators=500; min_samples_leaf=2",
    "ExtraTrees": "n_estimators=500; min_samples_leaf=2",
    "XGBoost": "n_estimators=500; depth=2; learning_rate=0.03",
    "CatBoost": "iterations=700; depth=3; learning_rate=0.03",
}

display(pd.DataFrame([
    {"model": name, "fixed_config": config}
    for name, config in SSCMP_MODEL_CONFIGS.items()
]))


In [ ]:
# Fit every requested Stage 2 model on both feature sets
def sscmp_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(y_pred)
    return {
        "original_r2": r2_score(y_true, y_pred),
        "original_rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "original_mae": float(mean_absolute_error(y_true, y_pred)),
        "log_r2": r2_score(y_true_log, y_pred_log),
        "log_rmse": float(np.sqrt(mean_squared_error(y_true_log, y_pred_log))),
        "spearman_r": float(spearmanr(y_true, y_pred).statistic),
        "bias": float(np.mean(y_pred - y_true)),
    }


sscmp_rows = []
sscmp_prediction_tables = {}

for feature_set_name, feature_cols in SSCMP_FEATURE_SETS.items():
    classifier = sscmp_classifier()
    classifier.fit(sscmp_train[feature_cols], sscmp_train["is_high"])
    p_high = classifier.predict_proba(sscmp_test[feature_cols])[:, 1]
    predicted_high = (p_high >= 0.5).astype(int)
    classifier_balanced_accuracy = balanced_accuracy_score(
        sscmp_test["is_high"], predicted_high
    )
    classifier_f1_high = f1_score(
        sscmp_test["is_high"], predicted_high, pos_label=1
    )

    for model_name, regressor in sscmp_regressors().items():
        regressor.fit(
            sscmp_high_train[feature_cols],
            sscmp_high_train["SS_log1p"],
        )
        high_pred_log = np.asarray(
            regressor.predict(sscmp_test[feature_cols])
        ).reshape(-1)
        high_pred_original = np.expm1(np.clip(high_pred_log, 0, None))
        final_prediction = (
            p_high * high_pred_original
            + (1.0 - p_high) * sscmp_low_value
        )

        metric = sscmp_metrics(
            sscmp_test[SSCMP_TARGET].to_numpy(dtype=float),
            final_prediction,
        )
        sscmp_rows.append({
            "feature_set": feature_set_name,
            "stage1_classifier": "LogisticRegression",
            "stage2_model": model_name,
            "fixed_config": SSCMP_MODEL_CONFIGS[model_name],
            "n_features": len(feature_cols),
            "n_train_stage1": len(sscmp_train),
            "n_train_stage2_high": len(sscmp_high_train),
            "n_test": len(sscmp_test),
            "classifier_balanced_accuracy": classifier_balanced_accuracy,
            "classifier_f1_high": classifier_f1_high,
            **metric,
        })
        sscmp_prediction_tables[(feature_set_name, model_name)] = pd.DataFrame({
            "sample_id": sscmp_test["sample_id"].values,
            "observed": sscmp_test[SSCMP_TARGET].values,
            "predicted": final_prediction,
            "residual": final_prediction - sscmp_test[SSCMP_TARGET].values,
            "p_high": p_high,
        })

ss_two_stage_model_comparison = (
    pd.DataFrame(sscmp_rows)
    .sort_values(["original_r2", "original_rmse"], ascending=[False, True])
    .reset_index(drop=True)
)

report_columns = [
    "feature_set", "stage2_model", "fixed_config",
    "n_features", "n_train_stage1", "n_train_stage2_high", "n_test",
    "original_r2", "original_rmse", "original_mae",
    "log_r2", "log_rmse", "spearman_r", "bias",
    "classifier_balanced_accuracy", "classifier_f1_high",
]

print("All fixed two-stage model results")
display(ss_two_stage_model_comparison[report_columns].round(4))


In [ ]:
# Reporting tables and model-comparison plots
ss_best_by_feature_set = (
    ss_two_stage_model_comparison
    .sort_values(["original_r2", "original_rmse"], ascending=[False, True])
    .groupby("feature_set", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Best fixed Stage 2 model within each feature set")
display(ss_best_by_feature_set[[
    "feature_set", "stage2_model", "fixed_config",
    "n_features", "n_train_stage1", "n_train_stage2_high", "n_test",
    "original_r2", "original_rmse", "original_mae",
    "log_r2", "log_rmse", "spearman_r", "bias",
]].round(4))

ss_report_pivot = (
    ss_two_stage_model_comparison
    .pivot(index="stage2_model", columns="feature_set", values=["original_r2", "original_rmse"])
)
ss_report_pivot.columns = [f"{metric}_{feature_set}" for metric, feature_set in ss_report_pivot.columns]
ss_report_pivot = ss_report_pivot.reset_index().sort_values(
    "original_r2_S2 + Context", ascending=False
)

print("Compact model-by-feature-set table")
display(ss_report_pivot.round(4))

model_order = list(SSCMP_MODEL_CONFIGS)
x = np.arange(len(model_order))
width = 0.38
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for offset, (feature_set_name, color) in zip(
    [-width / 2, width / 2],
    [("S2 only", "#377eb8"), ("S2 + Context", "#4daf4a")],
):
    ordered = (
        ss_two_stage_model_comparison
        .set_index(["feature_set", "stage2_model"])
        .loc[[(feature_set_name, model) for model in model_order]]
    )
    axes[0].bar(x + offset, ordered["original_r2"], width, label=feature_set_name, color=color)
    axes[1].bar(x + offset, ordered["original_rmse"], width, label=feature_set_name, color=color)

axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("Original-scale R2")
axes[0].set_title("SS two-stage model comparison: R2")
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

axes[1].set_ylabel("RMSE (mg/L)")
axes[1].set_title("SS two-stage model comparison: RMSE")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_order, rotation=30, ha="right")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


### Final Model Performance Summary

The table below reports every requested Stage 2 regression model under the same two-stage setup and ranks the results by original-scale R2. The best overall configuration is shown separately below the complete table.


In [ ]:
# SS_FINAL_REPORTING_SUMMARY_TABLE
ss_model_summary_table = (
    ss_two_stage_model_comparison
    .rename(columns={
        "stage2_model": "model",
        "feature_set": "feature set",
        "original_r2": "R2",
        "original_rmse": "RMSE",
        "original_mae": "MAE",
        "log_r2": "log R2",
        "log_rmse": "log RMSE",
        "spearman_r": "spearman r",
        "bias": "bias",
    })
    [[
        "model", "feature set", "R2", "RMSE", "MAE",
        "log R2", "log RMSE", "spearman r", "bias",
    ]]
    .sort_values(["R2", "RMSE"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Complete SS model comparison, ranked by original-scale R2")
display(ss_model_summary_table.round(4))

ss_best_overall_model = ss_model_summary_table.head(1).copy()
print("Best overall SS model")
display(ss_best_overall_model.round(4))


<!-- SS_TWO_STAGE_FIXED_MODEL_COMPARISON_END -->
